#Inicjalizacja środowiska i pobranie datasetow

## 1. Setup środowiska i Google Drive

Ta komórka:
- instaluje potrzebne biblioteki,
- montuje Google Drive,
- tworzy strukturę katalogów projektu,
- sprawdza, czy w Colabie aktywne jest GPU,
- ustawia podstawową konfigurację ścieżek i losowości.

W tej komórce nie analizujemy jeszcze datasetów — tylko przygotowujemy środowisko pracy.

In [ ]:
%%capture
!pip -q install -U sentence-transformers pyarrow scikit-learn tqdm openpyxl datasets huggingface_hub gdown

import os
import gc
import re
import io
import json
import shutil
import random
import zipfile
import warnings
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from IPython.display import display
from google.colab import drive

warnings.filterwarnings("ignore")

# ===== Losowość =====
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)

# ===== Montowanie Google Drive =====
drive.mount("/content/drive")

# ===== Struktura katalogów =====
PROJECT_ROOT = Path("/content/drive/MyDrive/Hallucination_Project")
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
ACTIVE_DIR = PROCESSED_DIR / "active"
EMB_DIR = PROJECT_ROOT / "embeddings"
RESULTS_DIR = PROJECT_ROOT / "results"
REPORTS_DIR = PROJECT_ROOT / "reports"
TMP_DIR = PROJECT_ROOT / "tmp"

for path in [RAW_DIR, PROCESSED_DIR, ACTIVE_DIR, EMB_DIR, RESULTS_DIR, REPORTS_DIR, TMP_DIR]:
    path.mkdir(parents=True, exist_ok=True)

# ===== GPU =====
if not torch.cuda.is_available():
    raise SystemError(
        "❌ Brak GPU. W Colabie ustaw: Środowisko wykonawcze -> Zmień typ środowiska -> GPU."
    )

DEVICE = "cuda"
torch.set_float32_matmul_precision("high")

print("✅ Środowisko gotowe")
print("👉 GPU:", torch.cuda.get_device_name(0))
print("👉 VRAM [GB]:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
print("👉 PROJECT_ROOT:", PROJECT_ROOT)
print("👉 RAW_DIR:", RAW_DIR)

## 2. Pobranie datasetów do katalogu `data/raw`

Ta komórka:
- pobiera publicznie dostępne datasety i repozytoria źródłowe,
- zapisuje je w uporządkowanej strukturze katalogów,
- pozwala wybrać profil pobierania:
  - `starter` — lżejszy wariant do szybkiego startu w Colabie,
  - `full` — pełniejsze pobranie, cięższe dyskowo i czasowo.

Domyślnie ustawiony jest profil `starter`, żeby nie przeciążyć darmowego Colaba już na początku.

In [ ]:
from huggingface_hub import snapshot_download

# =====================================================================================
# PROFILE POBIERANIA
# -------------------------------------------------------------------------------------
# "starter" -> lżejszy start do Colaba:
#   - HaluEval
#   - Med-HALT
#   - MedHal-LF (lżejsza wersja zamiast pełnego MedHal)
#   - MedHallu repo
#   - HaluQuestQA repo
#   - Med-MMHL repo + próba pobrania folderu z Dropbox
#
# "full" -> cięższy wariant:
#   - MedHal pełny
# =====================================================================================
DOWNLOAD_PROFILE = "starter"   # zmień na "full", jeśli chcesz pełny MedHal

# ===== Źródła =====
DATA_SOURCES = {
    "HaluEval": {
        "type": "git",
        "url": "https://github.com/RUCAIBox/HaluEval.git",
        "target_dir": RAW_DIR / "HaluEval",
    },
    "Med-HALT": {
        "type": "hf_dataset_snapshot",
        "repo_id": "openlifescienceai/Med-HALT",
        "target_dir": RAW_DIR / "Med-HALT",
    },
    "MedHal": {
        "type": "hf_dataset_snapshot",
        "repo_id": "GM07/medhal",
        "target_dir": RAW_DIR / "MedHal",
    },
    "MedHal-LF": {
        "type": "hf_dataset_snapshot",
        "repo_id": "GM07/medhal-lf",
        "target_dir": RAW_DIR / "MedHal-LF",
    },
    "MedHallu": {
        "type": "git",
        "url": "https://github.com/MedHallu/MedHallu.git",
        "target_dir": RAW_DIR / "MedHallu",
    },
    "HalluQuestQA": {
        "type": "git",
        "url": "https://github.com/UKPLab/acl2025-lfqa-hallucination.git",
        "target_dir": RAW_DIR / "HalluQuestQA",
    },
    "Med-MMHL": {
        "type": "git_plus_dropbox",
        "git_url": "https://github.com/styxsys0927/Med-MMHL.git",
        "dropbox_url": "https://www.dropbox.com/scl/fo/zvud6ta0uaqm2j1liupts/h?dl=1&rlkey=zhychubvhspdxramyjdqjteqd",
        "target_dir": RAW_DIR / "Med-MMHL",
    },
}

def run_cmd(cmd, cwd=None):
    print("▶", " ".join(cmd))
    result = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stdout)
        print(result.stderr)
        raise RuntimeError(f"Polecenie nie powiodło się: {' '.join(cmd)}")
    return result

def ensure_git_repo(url: str, target_dir: Path):
    if target_dir.exists() and any(target_dir.iterdir()):
        print(f"📦 Repo już istnieje: {target_dir}")
        return
    target_dir.parent.mkdir(parents=True, exist_ok=True)
    run_cmd(["git", "clone", "--depth", "1", url, str(target_dir)])

def ensure_hf_snapshot(repo_id: str, target_dir: Path):
    if target_dir.exists() and any(target_dir.iterdir()):
        print(f"📦 Snapshot już istnieje: {target_dir}")
        return
    target_dir.mkdir(parents=True, exist_ok=True)
    snapshot_download(
        repo_id=repo_id,
        repo_type="dataset",
        local_dir=str(target_dir),
        local_dir_use_symlinks=False,
        resume_download=True,
    )
    print(f"✅ Pobrano HF dataset: {repo_id} -> {target_dir}")

def try_dropbox_folder_zip(dropbox_url: str, target_dir: Path):
    """
    Próbuje pobrać folder Dropbox jako zip.
    Jeśli się nie uda, zostawia samo repo jako źródło pomocnicze.
    """
    target_dir.mkdir(parents=True, exist_ok=True)
    zip_path = TMP_DIR / "med_mmhl_dropbox.zip"
    extract_dir = target_dir / "dropbox_data"

    if extract_dir.exists() and any(extract_dir.iterdir()):
        print(f"📦 Dropbox data już istnieją: {extract_dir}")
        return

    try:
        print("⬇️ Próba pobrania Med-MMHL z Dropbox...")
        run_cmd(["wget", "-O", str(zip_path), dropbox_url])

        extract_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(extract_dir)

        print(f"✅ Rozpakowano Dropbox -> {extract_dir}")
    except Exception as e:
        print(f"⚠️ Nie udało się pobrać Dropbox ZIP dla Med-MMHL: {e}")
        print("   Repo zostało sklonowane, ale dane trzeba będzie ewentualnie dograć ręcznie.")
    finally:
        if zip_path.exists():
            zip_path.unlink(missing_ok=True)

download_plan = [
    "HaluEval",
    "Med-HALT",
    "HalluQuestQA",
    "Med-MMHL",
]

if DOWNLOAD_PROFILE == "starter":
    download_plan.append("MedHal-LF")
elif DOWNLOAD_PROFILE == "full":
    download_plan.append("MedHal")
else:
    raise ValueError("DOWNLOAD_PROFILE musi być jednym z: 'starter', 'full'")

download_status = []

for name in download_plan:
    cfg = DATA_SOURCES[name]
    target_dir = cfg["target_dir"]

    try:
        if cfg["type"] == "git":
            ensure_git_repo(cfg["url"], target_dir)

        elif cfg["type"] == "hf_dataset_snapshot":
            ensure_hf_snapshot(cfg["repo_id"], target_dir)

        elif cfg["type"] == "git_plus_dropbox":
            ensure_git_repo(cfg["git_url"], target_dir)
            try_dropbox_folder_zip(cfg["dropbox_url"], target_dir)

        download_status.append({
            "dataset": name,
            "status": "ok",
            "path": str(target_dir),
        })

    except Exception as e:
        download_status.append({
            "dataset": name,
            "status": f"error: {e}",
            "path": str(target_dir),
        })

download_status_df = pd.DataFrame(download_status)
display(download_status_df)

print("\n✅ Pobieranie zakończone.")
print("📌 Jeśli później będziesz chciał pełny MedHal, ustaw DOWNLOAD_PROFILE='full' i uruchom tę komórkę jeszcze raz.")

▶ git clone --depth 1 https://github.com/RUCAIBox/HaluEval.git /content/drive/MyDrive/Hallucination_Project/data/raw/HaluEval


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

✅ Pobrano HF dataset: openlifescienceai/Med-HALT -> /content/drive/MyDrive/Hallucination_Project/data/raw/Med-HALT
▶ git clone --depth 1 https://github.com/MedHallu/MedHallu.git /content/drive/MyDrive/Hallucination_Project/data/raw/MedHallu
▶ git clone --depth 1 https://github.com/UKPLab/acl2025-lfqa-hallucination.git /content/drive/MyDrive/Hallucination_Project/data/raw/HalluQuestQA
▶ git clone --depth 1 https://github.com/styxsys0927/Med-MMHL.git /content/drive/MyDrive/Hallucination_Project/data/raw/Med-MMHL
⬇️ Próba pobrania Med-MMHL z Dropbox...
▶ wget -O /content/drive/MyDrive/Hallucination_Project/tmp/med_mmhl_dropbox.zip https://www.dropbox.com/scl/fo/zvud6ta0uaqm2j1liupts/h?dl=1&rlkey=zhychubvhspdxramyjdqjteqd
✅ Rozpakowano Dropbox -> /content/drive/MyDrive/Hallucination_Project/data/raw/Med-MMHL/dropbox_data


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

✅ Pobrano HF dataset: GM07/medhal-lf -> /content/drive/MyDrive/Hallucination_Project/data/raw/MedHal-LF


,dataset,status,path
0,HaluEval,ok,/content/drive/MyDrive/Hallucination_Project/d...
1,Med-HALT,ok,/content/drive/MyDrive/Hallucination_Project/d...
2,MedHallu,ok,/content/drive/MyDrive/Hallucination_Project/d...
3,HalluQuestQA,ok,/content/drive/MyDrive/Hallucination_Project/d...
4,Med-MMHL,ok,/content/drive/MyDrive/Hallucination_Project/d...
5,MedHal-LF,ok,/content/drive/MyDrive/Hallucination_Project/d...



✅ Pobieranie zakończone.
📌 Jeśli później będziesz chciał pełny MedHal, ustaw DOWNLOAD_PROFILE='full' i uruchom tę komórkę jeszcze raz.


In [ ]:
from pathlib import Path
from google.colab import drive
import pandas as pd
import json
import os

# jeśli nie masz jeszcze:
# !pip -q install -U datasets pyarrow

from datasets import load_dataset

# ===== Bootstrap =====
drive.mount("/content/drive", force_remount=False)

PROJECT_ROOT = Path("/content/drive/MyDrive/Hallucination_Project")
RAW_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

MEDHALLU_RAW_DIR = RAW_DIR / "MedHallu_HF"
MEDHALLU_RAW_DIR.mkdir(parents=True, exist_ok=True)

# ===== Pobranie z HF =====
configs = ["pqa_labeled", "pqa_artificial"]
download_summary = []

for cfg in configs:
    print(f"\n=== Loading MedHallu config: {cfg} ===")
    ds = load_dataset("UTAustin-AIHealth/MedHallu", cfg)

    for split_name, split_ds in ds.items():
        out_dir = MEDHALLU_RAW_DIR / cfg
        out_dir.mkdir(parents=True, exist_ok=True)

        df = split_ds.to_pandas()
        out_path = out_dir / f"{split_name}.parquet"
        df.to_parquet(out_path, index=False)

        preview_path = out_dir / f"{split_name}_columns.json"
        with open(preview_path, "w", encoding="utf-8") as f:
            json.dump(
                {
                    "config": cfg,
                    "split": split_name,
                    "n_rows": int(len(df)),
                    "columns": list(df.columns),
                },
                f,
                indent=2,
                ensure_ascii=False,
            )

        download_summary.append({
            "config": cfg,
            "split": split_name,
            "rows": len(df),
            "saved_to": str(out_path),
            "columns": ", ".join(map(str, df.columns)),
        })

summary_df = pd.DataFrame(download_summary)
display(summary_df)

print("\n✅ MedHallu pobrany z Hugging Face i zapisany na Drive.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

=== Loading MedHallu config: pqa_labeled ===


README.md: 0.00B [00:00, ?B/s]

pqa_labeled/train-00000-of-00001.parquet:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]


=== Loading MedHallu config: pqa_artificial ===


pqa_artificial/train-00000-of-00001.parq(…):   0%|          | 0.00/10.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9000 [00:00<?, ? examples/s]

,config,split,rows,saved_to,columns
0,pqa_labeled,train,1000,/content/drive/MyDrive/Hallucination_Project/d...,"Question, Knowledge, Ground Truth, Difficulty ..."
1,pqa_artificial,train,9000,/content/drive/MyDrive/Hallucination_Project/d...,"Question, Knowledge, Ground Truth, Difficulty ..."



✅ MedHallu pobrany z Hugging Face i zapisany na Drive.


#Procesowanie datasetów

## Helpery do przetwarzania datasetów

Ta komórka definiuje:
- odczyt różnych formatów plików,
- elastyczny odczyt JSON / JSONL,
- funkcje do wyszukiwania kolumn po aliasach,
- normalizację tekstu,
- standardowy zapis do folderu per dataset.

Każdy dataset będzie potem przetwarzany osobną komórką do wspólnego formatu technicznego:
- `dataset`
- `subset`
- `sample_id`
- `question`
- `context`
- `response`
- `label`
- `has_question`
- `has_context`
- `task_type`
- `source_file`

Uwaga:
ten format jest wspólny technicznie, ale nie znaczy jeszcze, że każdy dataset jest bezpośrednio poprawny do wzoru SGI.

In [ ]:
import os
import re
import gc
import ast
import json
import numpy as np
import pandas as pd
from pathlib import Path

BY_DATASET_DIR = PROCESSED_DIR / "by_dataset"
BY_DATASET_DIR.mkdir(parents=True, exist_ok=True)

SUPPORTED_EXTENSIONS = {
    ".csv", ".tsv", ".json", ".jsonl", ".parquet", ".xlsx", ".xls",
    ".csv.gz", ".tsv.gz", ".json.gz", ".jsonl.gz"
}

def full_suffix(path: Path) -> str:
    return "".join(path.suffixes).lower()

def is_supported_file(path: Path) -> bool:
    suffix = full_suffix(path)
    return any(suffix.endswith(ext) for ext in SUPPORTED_EXTENSIONS)

def normalize_text(x) -> str:
    if x is None:
        return ""
    try:
        if pd.isna(x):
            return ""
    except Exception:
        pass
    text = str(x).strip()
    text = re.sub(r"\s+", " ", text)
    return text

def read_any_table(path: Path, nrows: int | None = None) -> pd.DataFrame:
    suffix = full_suffix(path)

    if suffix.endswith(".csv") or suffix.endswith(".csv.gz"):
        return pd.read_csv(path, nrows=nrows)

    if suffix.endswith(".tsv") or suffix.endswith(".tsv.gz"):
        return pd.read_csv(path, sep="\t", nrows=nrows)

    if suffix.endswith(".parquet"):
        df = pd.read_parquet(path)
        return df.head(nrows) if nrows else df

    if suffix.endswith(".jsonl") or suffix.endswith(".jsonl.gz"):
        df = pd.read_json(path, lines=True)
        return df.head(nrows) if nrows else df

    if suffix.endswith(".json") or suffix.endswith(".json.gz"):
        with open(path, "r", encoding="utf-8") as f:
            obj = json.load(f)
        if isinstance(obj, dict):
            if "data" in obj and isinstance(obj["data"], list):
                obj = obj["data"]
            elif "records" in obj and isinstance(obj["records"], list):
                obj = obj["records"]
            else:
                obj = [obj]
        df = pd.json_normalize(obj)
        return df.head(nrows) if nrows else df

    if suffix.endswith(".xlsx") or suffix.endswith(".xls"):
        df = pd.read_excel(path)
        return df.head(nrows) if nrows else df

    raise ValueError(f"Nieobsługiwany format: {path}")

def load_json_flex(path: Path):
    """
    Obsługuje:
    - klasyczny JSON list/dict
    - JSONL
    - przypadki typu 'wiele obiektów JSON jeden po drugim'
    """
    text = path.read_text(encoding="utf-8", errors="ignore").strip()

    # 1) normalny json
    try:
        obj = json.loads(text)
        if isinstance(obj, dict):
            return [obj]
        if isinstance(obj, list):
            return obj
    except Exception:
        pass

    # 2) JSONL
    rows = []
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            rows.append(json.loads(line))
        except Exception:
            pass
    if rows:
        return rows

    # 3) sekwencja wielu obiektów JSON
    rows = []
    decoder = json.JSONDecoder()
    idx = 0
    while idx < len(text):
        while idx < len(text) and text[idx].isspace():
            idx += 1
        if idx >= len(text):
            break
        try:
            obj, end = decoder.raw_decode(text, idx)
            rows.append(obj)
            idx = end
        except Exception:
            break

    if rows:
        return rows

    raise ValueError(f"Nie udało się sparsować JSON: {path}")

def find_column(df: pd.DataFrame, aliases: list[str]) -> str | None:
    norm = {str(c).strip().lower(): c for c in df.columns}

    for alias in aliases:
        alias_l = alias.lower()
        if alias_l in norm:
            return norm[alias_l]

    for alias in aliases:
        alias_l = alias.lower()
        for col_l, col in norm.items():
            if alias_l in col_l:
                return col

    return None

QUESTION_ALIASES = [
    "question", "query", "prompt", "user_query", "question_text", "title", "pmid", "abstract"
]
CONTEXT_ALIASES = [
    "context", "knowledge", "document", "article", "passage", "dialogue_history",
    "options", "choices", "evidence", "source_text", "body"
]
RESPONSE_ALIASES = [
    "response", "answer", "statement", "chatgpt_response", "summary",
    "right_answer", "right_response", "right_summary", "hallucinated_answer",
    "hallucinated_response", "hallucinated_summary", "ans1_text", "ans2_text",
    "target", "output", "link", "pubmedlink"
]
LABEL_ALIASES = [
    "label", "hallucination", "hallucination_label", "is_hallucination",
    "factuality_label", "target"
]
ID_ALIASES = ["id", "ID", "sample_id", "source_file", "inner_id"]

def to_python_list(x):
    if x is None:
        return []
    try:
        if pd.isna(x):
            return []
    except Exception:
        pass
    if isinstance(x, list):
        return x
    if isinstance(x, tuple):
        return list(x)
    if isinstance(x, str):
        s = x.strip()
        if not s:
            return []
        try:
            val = ast.literal_eval(s)
            if isinstance(val, list):
                return val
            return [val]
        except Exception:
            return [s]
    return [x]

def yes_no_to_hallucination(x):
    s = normalize_text(x).lower()
    if s in {"yes", "y", "true", "1", "hallucination", "hallucinated"}:
        return 1
    if s in {"no", "n", "false", "0", "non-hallucination", "correct", "factual", "supported"}:
        return 0
    return np.nan

def factual_bool_to_hallucination(x):
    """
    W MedHal True = factual, False = non-factual.
    Zmieniamy na:
    0 = non-hallucination
    1 = hallucination
    """
    if isinstance(x, (bool, np.bool_)):
        return 0 if bool(x) else 1
    s = normalize_text(x).lower()
    if s in {"true", "1", "factual", "supported", "correct"}:
        return 0
    if s in {"false", "0", "non-factual", "hallucination", "hallucinated", "unsupported", "incorrect"}:
        return 1
    return np.nan

def save_dataset_outputs(dataset_name: str, native_df: pd.DataFrame, notes: dict | None = None):
    out_dir = BY_DATASET_DIR / dataset_name
    out_dir.mkdir(parents=True, exist_ok=True)

    native_path = out_dir / "native.parquet"
    native_csv_path = out_dir / "native.csv"
    schema_path = out_dir / "schema_preview.csv"
    notes_path = out_dir / "notes.json"

    native_df.to_parquet(native_path, index=False)
    native_df.to_csv(native_csv_path, index=False)

    schema_preview = pd.DataFrame({
        "column": native_df.columns,
        "dtype": [str(native_df[c].dtype) for c in native_df.columns],
        "non_null": [int(native_df[c].notna().sum()) for c in native_df.columns],
    })
    schema_preview.to_csv(schema_path, index=False)

    if notes is None:
        notes = {}
    with open(notes_path, "w", encoding="utf-8") as f:
        json.dump(notes, f, indent=2, ensure_ascii=False)

    print(f"✅ Zapisano {dataset_name}")
    print("   ", native_path)
    print("   ", native_csv_path)
    print("   ", schema_path)
    print("   ", notes_path)

def make_standard_df(rows: list[dict]) -> pd.DataFrame:
    df = pd.DataFrame(rows)
    expected = [
        "dataset", "subset", "sample_id", "question", "context", "response",
        "label", "has_question", "has_context", "task_type", "source_file"
    ]
    for c in expected:
        if c not in df.columns:
            df[c] = np.nan

    # porządek
    left = expected
    right = [c for c in df.columns if c not in left]
    df = df[left + right].copy()

    # normalizacja tekstów
    for c in ["dataset", "subset", "sample_id", "question", "context", "response", "task_type", "source_file"]:
        df[c] = df[c].map(normalize_text)

    if "label" in df.columns:
        df["label"] = pd.to_numeric(df["label"], errors="coerce")

    df["has_question"] = df["has_question"].fillna(False).astype(bool)
    df["has_context"] = df["has_context"].fillna(False).astype(bool)

    return df

## HaluEval — przetwarzanie

HaluEval ma kilka różnych plików i różne pola w zależności od podzadania:
- `qa_data.json`
- `dialogue_data.json`
- `summarization_data.json`
- `general_data.json`

Dla QA / dialogue / summarization tworzymy po dwa rekordy:
- wersję poprawną,
- wersję halucynowaną.

Dla `general_data.json` bierzemy gotową odpowiedź ChatGPT i etykietę człowieka.

In [ ]:
root = RAW_DIR / "HaluEval"
if not root.exists():
    raise FileNotFoundError(f"Brak folderu: {root}")

files = {
    "qa": next(iter(root.rglob("qa_data.json")), None),
    "dialogue": next(iter(root.rglob("dialogue_data.json")), None),
    "summarization": next(iter(root.rglob("summarization_data.json")), None),
    "general": next(iter(root.rglob("general_data.json")), None),
}

rows = []

# QA
if files["qa"] is not None:
    items = load_json_flex(files["qa"])
    for i, x in enumerate(items):
        q = normalize_text(x.get("question"))
        c = normalize_text(x.get("knowledge"))
        right = normalize_text(x.get("right_answer"))
        hallu = normalize_text(x.get("hallucinated_answer"))
        sid = normalize_text(x.get("ID", f"qa_{i}"))

        if q and (right or hallu):
            if right:
                rows.append({
                    "dataset": "HaluEval",
                    "subset": "qa",
                    "sample_id": f"{sid}_right",
                    "question": q,
                    "context": c,
                    "response": right,
                    "label": 0,
                    "has_question": True,
                    "has_context": bool(c),
                    "task_type": "qa",
                    "source_file": str(files["qa"]),
                })
            if hallu:
                rows.append({
                    "dataset": "HaluEval",
                    "subset": "qa",
                    "sample_id": f"{sid}_hallu",
                    "question": q,
                    "context": c,
                    "response": hallu,
                    "label": 1,
                    "has_question": True,
                    "has_context": bool(c),
                    "task_type": "qa",
                    "source_file": str(files["qa"]),
                })

# Dialogue
if files["dialogue"] is not None:
    items = load_json_flex(files["dialogue"])
    for i, x in enumerate(items):
        q = normalize_text(x.get("dialogue_history"))
        c = normalize_text(x.get("knowledge"))
        right = normalize_text(x.get("right_response"))
        hallu = normalize_text(x.get("hallucinated_response"))
        sid = normalize_text(x.get("ID", f"dialogue_{i}"))

        if q and (right or hallu):
            if right:
                rows.append({
                    "dataset": "HaluEval",
                    "subset": "dialogue",
                    "sample_id": f"{sid}_right",
                    "question": q,
                    "context": c,
                    "response": right,
                    "label": 0,
                    "has_question": True,
                    "has_context": bool(c),
                    "task_type": "dialogue",
                    "source_file": str(files["dialogue"]),
                })
            if hallu:
                rows.append({
                    "dataset": "HaluEval",
                    "subset": "dialogue",
                    "sample_id": f"{sid}_hallu",
                    "question": q,
                    "context": c,
                    "response": hallu,
                    "label": 1,
                    "has_question": True,
                    "has_context": bool(c),
                    "task_type": "dialogue",
                    "source_file": str(files["dialogue"]),
                })

# Summarization
if files["summarization"] is not None:
    items = load_json_flex(files["summarization"])
    for i, x in enumerate(items):
        q = "Summarize the document faithfully."
        c = normalize_text(x.get("document"))
        right = normalize_text(x.get("right_summary"))
        hallu = normalize_text(x.get("hallucinated_summary"))
        sid = normalize_text(x.get("ID", f"summ_{i}"))

        if c and (right or hallu):
            if right:
                rows.append({
                    "dataset": "HaluEval",
                    "subset": "summarization",
                    "sample_id": f"{sid}_right",
                    "question": q,
                    "context": c,
                    "response": right,
                    "label": 0,
                    "has_question": True,
                    "has_context": True,
                    "task_type": "summarization",
                    "source_file": str(files["summarization"]),
                })
            if hallu:
                rows.append({
                    "dataset": "HaluEval",
                    "subset": "summarization",
                    "sample_id": f"{sid}_hallu",
                    "question": q,
                    "context": c,
                    "response": hallu,
                    "label": 1,
                    "has_question": True,
                    "has_context": True,
                    "task_type": "summarization",
                    "source_file": str(files["summarization"]),
                })

# General
if files["general"] is not None:
    items = load_json_flex(files["general"])
    for i, x in enumerate(items):
        q = normalize_text(x.get("user_query"))
        r = normalize_text(x.get("chatgpt_response"))
        y = yes_no_to_hallucination(x.get("hallucination") or x.get("hallucination_label"))
        sid = normalize_text(x.get("ID", f"general_{i}"))

        if q and r:
            rows.append({
                "dataset": "HaluEval",
                "subset": "general",
                "sample_id": sid,
                "question": q,
                "context": "",
                "response": r,
                "label": y,
                "has_question": True,
                "has_context": False,
                "task_type": "general_query",
                "source_file": str(files["general"]),
            })

halueval_df = make_standard_df(rows)
save_dataset_outputs(
    "HaluEval",
    halueval_df,
    notes={
        "sgi_ready_rows": int((halueval_df["has_question"] & halueval_df["has_context"]).sum()),
        "total_rows": int(len(halueval_df)),
    }
)

display(halueval_df.head(10))
print(halueval_df["subset"].value_counts(dropna=False))

✅ Zapisano HaluEval
    /content/drive/MyDrive/Hallucination_Project/data/processed/by_dataset/HaluEval/native.parquet
    /content/drive/MyDrive/Hallucination_Project/data/processed/by_dataset/HaluEval/native.csv
    /content/drive/MyDrive/Hallucination_Project/data/processed/by_dataset/HaluEval/schema_preview.csv
    /content/drive/MyDrive/Hallucination_Project/data/processed/by_dataset/HaluEval/notes.json


,dataset,subset,sample_id,question,context,response,label,has_question,has_context,task_type,source_file
0,HaluEval,qa,qa_0_right,Which magazine was started first Arthur's Maga...,Arthur's Magazine (1844–1846) was an American ...,Arthur's Magazine,0,True,True,qa,/content/drive/MyDrive/Hallucination_Project/d...
1,HaluEval,qa,qa_0_hallu,Which magazine was started first Arthur's Maga...,Arthur's Magazine (1844–1846) was an American ...,First for Women was started first.,1,True,True,qa,/content/drive/MyDrive/Hallucination_Project/d...
2,HaluEval,qa,qa_1_right,The Oberoi family is part of a hotel company t...,The Oberoi family is an Indian family that is ...,Delhi,0,True,True,qa,/content/drive/MyDrive/Hallucination_Project/d...
3,HaluEval,qa,qa_1_hallu,The Oberoi family is part of a hotel company t...,The Oberoi family is an Indian family that is ...,The Oberoi family's hotel company is based in ...,1,True,True,qa,/content/drive/MyDrive/Hallucination_Project/d...
4,HaluEval,qa,qa_2_right,Musician and satirist Allie Goertz wrote a son...,"Allison Beth ""Allie"" Goertz (born March 2, 199...",President Richard Nixon,0,True,True,qa,/content/drive/MyDrive/Hallucination_Project/d...
5,HaluEval,qa,qa_2_hallu,Musician and satirist Allie Goertz wrote a son...,"Allison Beth ""Allie"" Goertz (born March 2, 199...","Allie Goertz wrote a song about Milhouse, a po...",1,True,True,qa,/content/drive/MyDrive/Hallucination_Project/d...
6,HaluEval,qa,qa_3_right,What nationality was James Henry Miller's wife?,"Margaret ""Peggy"" Seeger (born June 17, 1935) i...",American,0,True,True,qa,/content/drive/MyDrive/Hallucination_Project/d...
7,HaluEval,qa,qa_3_hallu,What nationality was James Henry Miller's wife?,"Margaret ""Peggy"" Seeger (born June 17, 1935) i...",James Henry Miller's wife was British.,1,True,True,qa,/content/drive/MyDrive/Hallucination_Project/d...
8,HaluEval,qa,qa_4_right,Cadmium Chloride is slightly soluble in this c...,It is a hygroscopic solid that is highly solub...,alcohol,0,True,True,qa,/content/drive/MyDrive/Hallucination_Project/d...
9,HaluEval,qa,qa_4_hallu,Cadmium Chloride is slightly soluble in this c...,It is a hygroscopic solid that is highly solub...,water with a hint of alcohol,1,True,True,qa,/content/drive/MyDrive/Hallucination_Project/d...


subset
qa               20000
dialogue         20000
summarization    20000
general           4507
Name: count, dtype: int64


## MedHal — przetwarzanie

MedHal ma natywnie strukturę typu:
- `context`
- `statement`
- `label`
- `explanation`

To nie jest klasyczny triplet `question/context/response`, więc zapisujemy go do wspólnego formatu technicznego:
- `question` zostaje puste,
- `context` idzie do `context`,
- `statement` idzie do `response`,
- `label=True` oznacza sample factual, więc zamieniamy to na:
  - `0` = brak halucynacji
  - `1` = halucynacja

In [ ]:
import gc
import json
import shutil
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path

# =========================================================
# KONFIG
# =========================================================
MEDHAL_SOURCE_MODE = "auto"   # "auto", "lf_only", "full_only"
MEDHAL_CHUNK_SIZE = 10000
MEDHAL_MAX_ROWS = None        # np. 100000 do szybkiego testu; None = całość
MEDHAL_OVERWRITE_OUTPUT = True

# =========================================================
# WYBÓR ŹRÓDŁA
# =========================================================
source_priority = []

if MEDHAL_SOURCE_MODE == "auto":
    source_priority = ["MedHal-LF", "MedHal-LF-BAL", "MedHal"]
elif MEDHAL_SOURCE_MODE == "lf_only":
    source_priority = ["MedHal-LF", "MedHal-LF-BAL"]
elif MEDHAL_SOURCE_MODE == "full_only":
    source_priority = ["MedHal"]
else:
    raise ValueError("MEDHAL_SOURCE_MODE musi być jednym z: auto / lf_only / full_only")

candidate_roots = [RAW_DIR / name for name in source_priority if (RAW_DIR / name).exists()]
if not candidate_roots:
    raise FileNotFoundError("Nie znaleziono folderu MedHal / MedHal-LF zgodnie z wybranym MEDHAL_SOURCE_MODE.")

print("Wybrane źródła:")
for r in candidate_roots:
    print("-", r)

# =========================================================
# KATALOG WYJŚCIOWY
# =========================================================
out_dir = BY_DATASET_DIR / "MedHal"
if MEDHAL_OVERWRITE_OUTPUT and out_dir.exists():
    shutil.rmtree(out_dir)

out_dir.mkdir(parents=True, exist_ok=True)

csv_out = out_dir / "native.csv"
parquet_out = out_dir / "native.parquet"
schema_out = out_dir / "schema_preview.csv"
notes_out = out_dir / "notes.json"

# =========================================================
# HELPERY
# =========================================================
def medhal_find_candidate_files(roots):
    files = []
    for root in roots:
        for p in root.rglob("*"):
            if p.is_file() and is_supported_file(p):
                files.append(p)
    return sorted(set(files))

def medhal_detect_columns_from_preview(path: Path):
    """
    Odczytuje mały preview i szuka kluczowych kolumn.
    """
    try:
        preview = read_any_table(path, nrows=20)
    except Exception:
        return None

    c_col = find_column(preview, ["context"])
    s_col = find_column(preview, ["statement"])
    l_col = find_column(preview, ["label"])
    id_col = find_column(preview, ["id", "inner_id"])
    e_col = find_column(preview, ["explanation"])

    if c_col and s_col and l_col:
        return {
            "context": c_col,
            "statement": s_col,
            "label": l_col,
            "id": id_col,
            "explanation": e_col,
            "columns_preview": list(preview.columns),
        }
    return None

def iter_table_chunks(path: Path, usecols: list[str], chunksize: int = 10000):
    suffix = full_suffix(path)

    # CSV / TSV
    if suffix.endswith(".csv") or suffix.endswith(".csv.gz"):
        for chunk in pd.read_csv(path, usecols=lambda c: c in usecols, chunksize=chunksize):
            yield chunk
        return

    if suffix.endswith(".tsv") or suffix.endswith(".tsv.gz"):
        for chunk in pd.read_csv(path, sep="\t", usecols=lambda c: c in usecols, chunksize=chunksize):
            yield chunk
        return

    # Parquet
    if suffix.endswith(".parquet"):
        pf = pq.ParquetFile(path)
        existing_cols = [c for c in usecols if c in pf.schema.names]
        for batch in pf.iter_batches(batch_size=chunksize, columns=existing_cols):
            yield batch.to_pandas()
        return

    # JSONL
    if suffix.endswith(".jsonl") or suffix.endswith(".jsonl.gz"):
        for chunk in pd.read_json(path, lines=True, chunksize=chunksize):
            existing_cols = [c for c in usecols if c in chunk.columns]
            yield chunk[existing_cols].copy()
        return

    # JSON / XLSX fallback — jednorazowo, bez chunków
    # Tego wariantu lepiej unikać dla dużych plików, ale zostawiam jako awaryjny.
    df = read_any_table(path)
    existing_cols = [c for c in usecols if c in df.columns]
    yield df[existing_cols].copy()

def append_chunk_outputs(df_chunk: pd.DataFrame, parquet_writer_obj):
    """
    Dopisuje chunk do CSV i Parquet.
    """
    write_header = not csv_out.exists()
    df_chunk.to_csv(csv_out, mode="a", header=write_header, index=False)

    table = pa.Table.from_pandas(df_chunk, preserve_index=False)
    if parquet_writer_obj is None:
        parquet_writer_obj = pq.ParquetWriter(parquet_out, table.schema)
    parquet_writer_obj.write_table(table)

    return parquet_writer_obj

# =========================================================
# ODKRYCIE PLIKÓW MEDHAL
# =========================================================
candidate_files = medhal_find_candidate_files(candidate_roots)
if not candidate_files:
    raise FileNotFoundError("Nie znaleziono żadnych plików tabelarycznych dla MedHal.")

detected_files = []
schema_rows = []

for path in candidate_files:
    info = medhal_detect_columns_from_preview(path)
    if info is not None:
        detected_files.append((path, info))
        schema_rows.append({
            "file": str(path),
            "context_col": info["context"],
            "statement_col": info["statement"],
            "label_col": info["label"],
            "id_col": info["id"],
            "explanation_col": info["explanation"],
            "columns_preview": " | ".join(map(str, info["columns_preview"])),
        })

if not detected_files:
    raise ValueError("Nie znaleziono plików MedHal z kolumnami context / statement / label.")

schema_df = pd.DataFrame(schema_rows)
schema_df.to_csv(schema_out, index=False)
display(schema_df)

# =========================================================
# PRZETWARZANIE CHUNKAMI
# =========================================================
total_written = 0
parquet_writer = None
first_preview = None

for path, info in detected_files:
    subset_name = path.parent.name if path.parent != path.parent.parent else path.stem

    usecols = [info["context"], info["statement"], info["label"]]
    if info["id"] is not None:
        usecols.append(info["id"])
    if info["explanation"] is not None:
        usecols.append(info["explanation"])

    print(f"\nPrzetwarzanie: {path}")
    print("Kolumny:", usecols)

    for chunk_idx, chunk in enumerate(iter_table_chunks(path, usecols=usecols, chunksize=MEDHAL_CHUNK_SIZE), start=1):
        rows = []

        for i, row in chunk.iterrows():
            sid = normalize_text(row[info["id"]]) if info["id"] is not None and info["id"] in chunk.columns else f"{path.stem}_{chunk_idx}_{i}"
            c = normalize_text(row[info["context"]]) if info["context"] in chunk.columns else ""
            r = normalize_text(row[info["statement"]]) if info["statement"] in chunk.columns else ""
            y = factual_bool_to_hallucination(row[info["label"]]) if info["label"] in chunk.columns else np.nan
            explanation = normalize_text(row[info["explanation"]]) if info["explanation"] is not None and info["explanation"] in chunk.columns else ""

            rows.append({
                "dataset": "MedHal",
                "subset": subset_name,
                "sample_id": sid,
                "question": "",
                "context": c,
                "response": r,
                "label": y,
                "has_question": False,
                "has_context": bool(c),
                "task_type": "factual_statement_classification",
                "source_file": str(path),
                "explanation": explanation,
            })

        out_chunk = make_standard_df(rows)

        # odrzuć puste rekordy
        out_chunk = out_chunk[
            (out_chunk["context"].astype(str).str.strip() != "") &
            (out_chunk["response"].astype(str).str.strip() != "")
        ].reset_index(drop=True)

        if len(out_chunk) == 0:
            del chunk, rows, out_chunk
            gc.collect()
            continue

        if first_preview is None:
            first_preview = out_chunk.head(5).copy()

        # ograniczenie do testów
        if MEDHAL_MAX_ROWS is not None:
            remaining = MEDHAL_MAX_ROWS - total_written
            if remaining <= 0:
                break
            if len(out_chunk) > remaining:
                out_chunk = out_chunk.iloc[:remaining].copy()

        parquet_writer = append_chunk_outputs(out_chunk, parquet_writer)
        total_written += len(out_chunk)

        print(f"  chunk={chunk_idx} | zapisano {len(out_chunk)} rekordów | łącznie={total_written}")

        del chunk, rows, out_chunk
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        if MEDHAL_MAX_ROWS is not None and total_written >= MEDHAL_MAX_ROWS:
            break

    if MEDHAL_MAX_ROWS is not None and total_written >= MEDHAL_MAX_ROWS:
        break

if parquet_writer is not None:
    parquet_writer.close()

notes = {
    "warning": "MedHal nie jest natywnie question-context-response; question pozostaje puste.",
    "source_mode": MEDHAL_SOURCE_MODE,
    "chunk_size": MEDHAL_CHUNK_SIZE,
    "max_rows": MEDHAL_MAX_ROWS,
    "total_rows_saved": int(total_written),
    "output_csv": str(csv_out),
    "output_parquet": str(parquet_out),
}

with open(notes_out, "w", encoding="utf-8") as f:
    json.dump(notes, f, indent=2, ensure_ascii=False)

print("\n✅ MedHal zapisany bez trzymania całości w RAM")
print("CSV:", csv_out)
print("PARQUET:", parquet_out)
print("ROWS:", total_written)

if first_preview is not None:
    display(first_preview)

Wybrane źródła:
- /content/drive/MyDrive/Hallucination_Project/data/raw/MedHal-LF


,file,context_col,statement_col,label_col,id_col,explanation_col,columns_preview
0,/content/drive/MyDrive/Hallucination_Project/d...,context,statement,label,id,explanation,id | inner_id | source | synthetic | context |...
1,/content/drive/MyDrive/Hallucination_Project/d...,context,statement,label,id,explanation,id | inner_id | source | synthetic | context |...
2,/content/drive/MyDrive/Hallucination_Project/d...,context,statement,label,id,explanation,id | inner_id | source | synthetic | context |...
3,/content/drive/MyDrive/Hallucination_Project/d...,context,statement,label,id,explanation,id | inner_id | source | synthetic | context |...
4,/content/drive/MyDrive/Hallucination_Project/d...,context,statement,label,id,explanation,id | inner_id | source | synthetic | context |...
5,/content/drive/MyDrive/Hallucination_Project/d...,context,statement,label,id,explanation,id | inner_id | source | synthetic | context |...
6,/content/drive/MyDrive/Hallucination_Project/d...,context,statement,label,id,explanation,id | inner_id | source | synthetic | context |...



Przetwarzanie: /content/drive/MyDrive/Hallucination_Project/data/raw/MedHal-LF/data/train-00000-of-00007.parquet
Kolumny: ['context', 'statement', 'label', 'id', 'explanation']
  chunk=1 | zapisano 10000 rekordów | łącznie=10000
  chunk=2 | zapisano 10000 rekordów | łącznie=20000
  chunk=3 | zapisano 10000 rekordów | łącznie=30000
  chunk=4 | zapisano 10000 rekordów | łącznie=40000
  chunk=5 | zapisano 10000 rekordów | łącznie=50000
  chunk=6 | zapisano 10000 rekordów | łącznie=60000
  chunk=7 | zapisano 10000 rekordów | łącznie=70000
  chunk=8 | zapisano 10000 rekordów | łącznie=80000
  chunk=9 | zapisano 10000 rekordów | łącznie=90000
  chunk=10 | zapisano 10000 rekordów | łącznie=100000
  chunk=11 | zapisano 10000 rekordów | łącznie=110000
  chunk=12 | zapisano 270 rekordów | łącznie=110270

Przetwarzanie: /content/drive/MyDrive/Hallucination_Project/data/raw/MedHal-LF/data/train-00001-of-00007.parquet
Kolumny: ['context', 'statement', 'label', 'id', 'explanation']
  chunk=1 | zapi

,dataset,subset,sample_id,question,context,response,label,has_question,has_context,task_type,source_file,explanation
0,MedHal,data,254edfd9-dd8c-49cd-803f-aeb97a7f783e,,A 24-year-old healthy woman presented with dif...,The patient is admitted to the Emergency depar...,0,False,True,factual_statement_classification,/content/drive/MyDrive/Hallucination_Project/d...,The statement is factual.
1,MedHal,data,a30de6a8-586f-4020-8ee4-e167f898a572,,A 24-year-old healthy woman presented with dif...,The patient is being cared for at the District...,1,False,True,factual_statement_classification,/content/drive/MyDrive/Hallucination_Project/d...,The patient is admitted to the Emergency depar...
2,MedHal,data,b182c8ca-feff-4d1f-a552-05f92b86a3a7,,A 24-year-old healthy woman presented with dif...,The patient is admitted for blunt nasal trauma...,0,False,True,factual_statement_classification,/content/drive/MyDrive/Hallucination_Project/d...,The statement is factual.
3,MedHal,data,e5a52314-7c68-4d71-8508-412afb04b047,,A 24-year-old healthy woman presented with dif...,The patient is admitted due to hemopneumothora...,1,False,True,factual_statement_classification,/content/drive/MyDrive/Hallucination_Project/d...,The patient is admitted for blunt nasal trauma...
4,MedHal,data,8a6cd51f-6c50-4b64-9d1a-b62dd98f0c70,,A 24-year-old healthy woman presented with dif...,"An MRI of the brain, cervical and thoracic spi...",0,False,True,factual_statement_classification,/content/drive/MyDrive/Hallucination_Project/d...,The statement is factual.


## MedHal — dopisanie syntetycznego pytania do już przetworzonego pliku

Ta komórka:
- otwiera już istniejący przetworzony MedHal z folderu `by_dataset/MedHal`,
- nie przetwarza datasetu od nowa,
- uzupełnia pole `question` stałym promptem:

> Based only on the clinical context, which medical statement is best supported?

- ustawia `has_question=True`,
- dodaje kolumny:
  - `question_mode`
  - `sgi_setting`

Przed nadpisaniem tworzy kopię zapasową starych plików.

In [ ]:
from pathlib import Path
from google.colab import drive
import pandas as pd
import json
import shutil
from datetime import datetime

# ===== Self-contained bootstrap =====
drive.mount("/content/drive", force_remount=False)

PROJECT_ROOT = Path("/content/drive/MyDrive/Hallucination_Project")
BY_DATASET_DIR = PROJECT_ROOT / "data" / "processed" / "by_dataset"

medhal_dir = BY_DATASET_DIR / "MedHal"
parquet_path = medhal_dir / "native.parquet"
csv_path = medhal_dir / "native.csv"
notes_path = medhal_dir / "notes.json"

if not medhal_dir.exists():
    raise FileNotFoundError(f"Brak folderu: {medhal_dir}")

if not parquet_path.exists() and not csv_path.exists():
    raise FileNotFoundError("Nie znaleziono ani native.parquet, ani native.csv dla MedHal.")

SYNTHETIC_QUESTION = "Based only on the clinical context, which medical statement is best supported?"

# ===== Wczytanie istniejącego pliku =====
if parquet_path.exists():
    df = pd.read_parquet(parquet_path)
    source_used = "parquet"
else:
    df = pd.read_csv(csv_path)
    source_used = "csv"

print(f"✅ Wczytano MedHal z: {source_used}")
print("Liczba rekordów:", len(df))

# ===== Backup przed edycją =====
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup_dir = medhal_dir / f"backup_before_question_patch_{timestamp}"
backup_dir.mkdir(parents=True, exist_ok=True)

if parquet_path.exists():
    shutil.copy2(parquet_path, backup_dir / "native.parquet")
if csv_path.exists():
    shutil.copy2(csv_path, backup_dir / "native.csv")
if notes_path.exists():
    shutil.copy2(notes_path, backup_dir / "notes.json")

print("💾 Backup zapisany w:", backup_dir)

# ===== Edycja istniejących danych =====
if "question" not in df.columns:
    df["question"] = ""
if "has_question" not in df.columns:
    df["has_question"] = False

df["question"] = SYNTHETIC_QUESTION
df["has_question"] = True
df["question_mode"] = "synthetic_global_template"
df["sgi_setting"] = "extended"

# opcjonalnie zostawiamy flagę metodologiczną
df["sgi_compatible_after_patch"] = (
    df["question"].astype(str).str.strip().ne("") &
    df["context"].astype(str).str.strip().ne("") &
    df["response"].astype(str).str.strip().ne("") &
    df["label"].notna()
)

# ===== Zapis nadpisujący istniejące pliki =====
df.to_parquet(parquet_path, index=False)
df.to_csv(csv_path, index=False)

# ===== Aktualizacja notes.json =====
notes = {}
if notes_path.exists():
    try:
        with open(notes_path, "r", encoding="utf-8") as f:
            notes = json.load(f)
    except Exception:
        notes = {}

notes["question_patch_applied"] = True
notes["question_patch_timestamp"] = timestamp
notes["synthetic_question"] = SYNTHETIC_QUESTION
notes["question_mode"] = "synthetic_global_template"
notes["sgi_setting"] = "extended"
notes["warning"] = (
    "MedHal nie ma natywnego question. Dodano syntetyczny globalny prompt "
    "dla eksperymentu SGI-extended."
)

with open(notes_path, "w", encoding="utf-8") as f:
    json.dump(notes, f, indent=2, ensure_ascii=False)

print("\n✅ Patch zakończony")
print("Parquet:", parquet_path)
print("CSV    :", csv_path)
print("Notes  :", notes_path)

display(df.head(5)[[
    "dataset", "subset", "sample_id", "question", "context", "response",
    "label", "has_question", "question_mode", "sgi_setting"
]])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Wczytano MedHal z: parquet
Liczba rekordów: 532558
💾 Backup zapisany w: /content/drive/MyDrive/Hallucination_Project/data/processed/by_dataset/MedHal/backup_before_question_patch_20260321_174156

✅ Patch zakończony
Parquet: /content/drive/MyDrive/Hallucination_Project/data/processed/by_dataset/MedHal/native.parquet
CSV    : /content/drive/MyDrive/Hallucination_Project/data/processed/by_dataset/MedHal/native.csv
Notes  : /content/drive/MyDrive/Hallucination_Project/data/processed/by_dataset/MedHal/notes.json


,dataset,subset,sample_id,question,context,response,label,has_question,question_mode,sgi_setting
0,MedHal,data,254edfd9-dd8c-49cd-803f-aeb97a7f783e,"Based only on the clinical context, which medi...",A 24-year-old healthy woman presented with dif...,The patient is admitted to the Emergency depar...,0,True,synthetic_global_template,extended
1,MedHal,data,a30de6a8-586f-4020-8ee4-e167f898a572,"Based only on the clinical context, which medi...",A 24-year-old healthy woman presented with dif...,The patient is being cared for at the District...,1,True,synthetic_global_template,extended
2,MedHal,data,b182c8ca-feff-4d1f-a552-05f92b86a3a7,"Based only on the clinical context, which medi...",A 24-year-old healthy woman presented with dif...,The patient is admitted for blunt nasal trauma...,0,True,synthetic_global_template,extended
3,MedHal,data,e5a52314-7c68-4d71-8508-412afb04b047,"Based only on the clinical context, which medi...",A 24-year-old healthy woman presented with dif...,The patient is admitted due to hemopneumothora...,1,True,synthetic_global_template,extended
4,MedHal,data,8a6cd51f-6c50-4b64-9d1a-b62dd98f0c70,"Based only on the clinical context, which medi...",A 24-year-old healthy woman presented with dif...,"An MRI of the brain, cervical and thoracic spi...",0,True,synthetic_global_template,extended


## Med-HALT — przetwarzanie

Med-HALT nie jest jednym binarnym zbiorem QA. To benchmark z wieloma podtestami:
- reasoning: FCT / fake / NOTA
- memory-based: abstract→link, PMID→title, title→link, link→title

Dlatego tutaj:
- czytamy każdy podzbiór osobno,
- normalizujemy nazwy kolumn,
- zapisujemy wszystko do wspólnego formatu technicznego best-effort,
- nie zakładamy z góry istnienia binarnej etykiety halucynacji.

In [ ]:
root = RAW_DIR / "Med-HALT"
if not root.exists():
    raise FileNotFoundError(f"Brak folderu: {root}")

subset_csvs = sorted(root.rglob("*.csv"))
if not subset_csvs:
    raise FileNotFoundError("Nie znaleziono plików CSV dla Med-HALT")

rows = []
schema_rows = []

for path in subset_csvs:
    df = pd.read_csv(path)
    subset = path.parent.name if path.parent.name != root.name else path.stem

    schema_rows.append({
        "subset": subset,
        "file": str(path),
        "n_rows": len(df),
        "columns": " | ".join(map(str, df.columns)),
    })

    q_col = find_column(df, ["question", "prompt", "query", "abstract", "pmid", "title", "link"])
    c_col = find_column(df, ["context", "options", "choices", "article", "document", "passage"])
    r_col = find_column(df, ["answer", "target", "output", "pubmedlink", "title", "link"])
    l_col = find_column(df, ["label", "hallucination", "target"])
    id_col = find_column(df, ["id", "ID"])

    for i, row in df.iterrows():
        sid = normalize_text(row[id_col]) if id_col else f"{subset}_{i}"

        q = normalize_text(row[q_col]) if q_col else ""
        c = normalize_text(row[c_col]) if c_col else ""
        r = normalize_text(row[r_col]) if r_col else ""

        # jeśli nic nie znaleźliśmy jako response, zlepimy niepuste kolumny wejściowe
        if not r:
            non_empty_parts = []
            for col in df.columns[:6]:
                val = normalize_text(row[col])
                if val:
                    non_empty_parts.append(f"{col}: {val}")
            r = " || ".join(non_empty_parts)

        y = np.nan
        if l_col:
            y = yes_no_to_hallucination(row[l_col])
            if pd.isna(y):
                y = factual_bool_to_hallucination(row[l_col])

        rows.append({
            "dataset": "Med-HALT",
            "subset": subset,
            "sample_id": sid,
            "question": q,
            "context": c,
            "response": r,
            "label": y,
            "has_question": bool(q),
            "has_context": bool(c),
            "task_type": "benchmark_reasoning_or_memory",
            "source_file": str(path),
        })

medhalt_df = make_standard_df(rows)
save_dataset_outputs(
    "Med-HALT",
    medhalt_df,
    notes={
        "warning": "Med-HALT to benchmark wielozadaniowy; zapis jest best-effort i nie oznacza pełnej SGI-zgodności.",
        "n_subsets": len(subset_csvs),
        "total_rows": int(len(medhalt_df)),
    }
)

schema_df = pd.DataFrame(schema_rows)
schema_df.to_csv(BY_DATASET_DIR / "Med-HALT" / "subset_schema.csv", index=False)

display(schema_df)
display(medhalt_df.head(10))

✅ Zapisano Med-HALT
    /content/drive/MyDrive/Hallucination_Project/data/processed/by_dataset/Med-HALT/native.parquet
    /content/drive/MyDrive/Hallucination_Project/data/processed/by_dataset/Med-HALT/native.csv
    /content/drive/MyDrive/Hallucination_Project/data/processed/by_dataset/Med-HALT/schema_preview.csv
    /content/drive/MyDrive/Hallucination_Project/data/processed/by_dataset/Med-HALT/notes.json


,subset,file,n_rows,columns
0,IR_abstract2pubmedlink,/content/drive/MyDrive/Hallucination_Project/d...,4916,PMID | Title | Authors | First Author | DOI | ...
1,IR_pmid2title,/content/drive/MyDrive/Hallucination_Project/d...,4916,PMID | Title | Authors | First Author | DOI | ...
2,IR_pubmedlink2title,/content/drive/MyDrive/Hallucination_Project/d...,4916,PMID | Title | Authors | First Author | DOI | ...
3,IR_title2pubmedlink,/content/drive/MyDrive/Hallucination_Project/d...,4916,PMID | Title | Authors | First Author | DOI | ...
4,reasoning_FCT,/content/drive/MyDrive/Hallucination_Project/d...,18866,id | dataset | question | options | correct_an...
5,reasoning_fake,/content/drive/MyDrive/Hallucination_Project/d...,1858,id | subject_name | topic_name | split_type | ...
6,reasoning_nota,/content/drive/MyDrive/Hallucination_Project/d...,18866,id | dataset | question | options | correct_an...


,dataset,subset,sample_id,question,context,response,label,has_question,has_context,task_type,source_file
0,Med-HALT,IR_abstract2pubmedlink,e7ab3a96-034b-4cc2-a4ee-a780c3e6dbd6,Background: The power of cosmic unicorn horns ...,,A Study on the Acoustic Manipulation of Quantu...,NaN,True,False,benchmark_reasoning_or_memory,/content/drive/MyDrive/Hallucination_Project/d...
1,Med-HALT,IR_abstract2pubmedlink,358bc4ea-6848-4997-b627-a008106927a3,Introducing technology to support patient care...,,The Inpatient Nursing Leadership Perspective o...,NaN,True,False,benchmark_reasoning_or_memory,/content/drive/MyDrive/Hallucination_Project/d...
2,Med-HALT,IR_abstract2pubmedlink,873eefa4-26d7-4ce3-ab97-934778d2cfa5,"Recently, light emitting diodes (LEDs) have be...",,Effects of light emitting diode irradiation on...,NaN,True,False,benchmark_reasoning_or_memory,/content/drive/MyDrive/Hallucination_Project/d...
3,Med-HALT,IR_abstract2pubmedlink,23f3a487-77f4-45aa-9d9c-4a10e54ea72f,Background: The role of flowers in quantum cyb...,,The Power of Stellar Dandelions in Shaping Qua...,NaN,True,False,benchmark_reasoning_or_memory,/content/drive/MyDrive/Hallucination_Project/d...
4,Med-HALT,IR_abstract2pubmedlink,824f8b20-44bf-4e8f-a8a3-9f3c287b97ce,Myelination is one of the most remarkable biol...,,Myelinating cocultures of rodent stem cell lin...,NaN,True,False,benchmark_reasoning_or_memory,/content/drive/MyDrive/Hallucination_Project/d...
5,Med-HALT,IR_abstract2pubmedlink,b71fc145-6cd2-43b8-939d-f9f43ec34b10,BACKGROUND: Machine learning (ML) methods have...,,Comparative Analysis of the Ability of Machine...,NaN,True,False,benchmark_reasoning_or_memory,/content/drive/MyDrive/Hallucination_Project/d...
6,Med-HALT,IR_abstract2pubmedlink,b1093fce-67a3-4c45-aca4-0cc99aabdfc2,"Background: In the land of undead, there exist...",,Unleashing the Magic of Necromancy: A Whimsica...,NaN,True,False,benchmark_reasoning_or_memory,/content/drive/MyDrive/Hallucination_Project/d...
7,Med-HALT,IR_abstract2pubmedlink,aebc0c63-3ac6-43a3-979c-d0354173e1cd,Juxtaglomerular cells (JGCs) of the olfactory ...,,Quantitative Association of Anatomical and Fun...,NaN,True,False,benchmark_reasoning_or_memory,/content/drive/MyDrive/Hallucination_Project/d...
8,Med-HALT,IR_abstract2pubmedlink,b55dac71-e0af-4358-8c08-2dc9adc21db6,We have generated a human induced pluripotent ...,,Generation of a human induced pluripotent stem...,NaN,True,False,benchmark_reasoning_or_memory,/content/drive/MyDrive/Hallucination_Project/d...
9,Med-HALT,IR_abstract2pubmedlink,2b85b5dd-0d65-4a8f-a393-b0be60cd1a30,OBJECTIVE: The objective of this study was to ...,,Twelve-Month Analysis of Nonattendance for Ini...,NaN,True,False,benchmark_reasoning_or_memory,/content/drive/MyDrive/Hallucination_Project/d...


## MedHallu — przetwarzanie

Publiczny opis MedHallu potwierdza, że jest to zbiór 10k par pytanie–odpowiedź z odpowiedziami halucynowanymi oraz poziomami trudności.
Ponieważ publiczny opis repo nie pokazuje wprost jednej stabilnej listy kolumn, ta komórka:
- przeszukuje repozytorium,
- wybiera najbardziej prawdopodobny plik tabelaryczny,
- mapuje kolumny po aliasach,
- jeśli znajdzie parę `correct_answer` / `hallucinated_answer`, tworzy rekordy analogicznie do HaluEval,
- w przeciwnym razie używa znalezionych kolumn `question/answer/label`.

In [ ]:
from pathlib import Path
from google.colab import drive
import pandas as pd
import numpy as np
import json
import shutil
from datetime import datetime

drive.mount("/content/drive", force_remount=False)

PROJECT_ROOT = Path("/content/drive/MyDrive/Hallucination_Project")
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
BY_DATASET_DIR = PROCESSED_DIR / "by_dataset"

MEDHALLU_RAW_DIR = RAW_DIR / "MedHallu_HF"
OUT_DIR = BY_DATASET_DIR / "MedHallu"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
BY_DATASET_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

if not MEDHALLU_RAW_DIR.exists():
    raise FileNotFoundError(
        f"Brak folderu {MEDHALLU_RAW_DIR}. Najpierw uruchom komórkę pobierającą MedHallu z Hugging Face."
    )

def normalize_text(x) -> str:
    if x is None:
        return ""
    try:
        if pd.isna(x):
            return ""
    except Exception:
        pass
    return " ".join(str(x).strip().split())

def join_knowledge(x) -> str:
    if x is None:
        return ""
    try:
        if pd.isna(x):
            return ""
    except Exception:
        pass

    if isinstance(x, list):
        parts = [normalize_text(v) for v in x if normalize_text(v)]
        return "\n".join(parts)

    return normalize_text(x)

def make_standard_df(rows: list[dict]) -> pd.DataFrame:
    df = pd.DataFrame(rows)
    expected = [
        "dataset", "subset", "sample_id", "question", "context", "response",
        "label", "has_question", "has_context", "task_type", "source_file"
    ]
    for c in expected:
        if c not in df.columns:
            df[c] = np.nan

    ordered = expected + [c for c in df.columns if c not in expected]
    df = df[ordered].copy()

    text_cols = ["dataset", "subset", "sample_id", "question", "context", "response", "task_type", "source_file"]
    for c in text_cols:
        df[c] = df[c].map(normalize_text)

    df["label"] = pd.to_numeric(df["label"], errors="coerce")
    df["has_question"] = df["has_question"].fillna(False).astype(bool)
    df["has_context"] = df["has_context"].fillna(False).astype(bool)
    return df

# backup starych wyników, jeśli są
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
for fname in ["native.parquet", "native.csv", "schema_preview.csv", "notes.json"]:
    src = OUT_DIR / fname
    if src.exists():
        backup_dir = OUT_DIR / f"backup_before_fix_{timestamp}"
        backup_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, backup_dir / fname)

parquet_files = sorted(MEDHALLU_RAW_DIR.rglob("*.parquet"))
if not parquet_files:
    raise FileNotFoundError(f"Nie znaleziono plików parquet w {MEDHALLU_RAW_DIR}")

rows = []
schema_rows = []

required_cols = [
    "Question",
    "Knowledge",
    "Ground Truth",
    "Hallucinated Answer",
    "Difficulty Level",
    "Category of Hallucination",
]

for path in parquet_files:
    df = pd.read_parquet(path)

    schema_rows.append({
        "file": str(path),
        "rows": len(df),
        "columns": " | ".join(map(str, df.columns)),
        "has_required_cols": all(c in df.columns for c in required_cols),
    })

    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        print(f"⚠️ Pomijam {path.name}: brak kolumn {missing}")
        continue

    subset_name = f"{path.parent.name}/{path.stem}"

    for i, row in df.iterrows():
        q = normalize_text(row["Question"])
        c = join_knowledge(row["Knowledge"])
        gt = normalize_text(row["Ground Truth"])
        hallu = normalize_text(row["Hallucinated Answer"])
        difficulty = normalize_text(row["Difficulty Level"])
        category = normalize_text(row["Category of Hallucination"])

        base_id = f"{path.parent.name}_{path.stem}_{i}"

        if q and gt:
            rows.append({
                "dataset": "MedHallu",
                "subset": difficulty if difficulty else subset_name,
                "sample_id": f"{base_id}_right",
                "question": q,
                "context": c,
                "response": gt,
                "label": 0,
                "has_question": True,
                "has_context": bool(c),
                "task_type": "medical_qa",
                "source_file": str(path),
                "difficulty": difficulty,
                "hallucination_category": category,
                "source_config": path.parent.name,
            })

        if q and hallu:
            rows.append({
                "dataset": "MedHallu",
                "subset": difficulty if difficulty else subset_name,
                "sample_id": f"{base_id}_hallu",
                "question": q,
                "context": c,
                "response": hallu,
                "label": 1,
                "has_question": True,
                "has_context": bool(c),
                "task_type": "medical_qa",
                "source_file": str(path),
                "difficulty": difficulty,
                "hallucination_category": category,
                "source_config": path.parent.name,
            })

medhallu_df = make_standard_df(rows)

native_parquet = OUT_DIR / "native.parquet"
native_csv = OUT_DIR / "native.csv"
schema_csv = OUT_DIR / "schema_preview.csv"
notes_json = OUT_DIR / "notes.json"

medhallu_df.to_parquet(native_parquet, index=False)
medhallu_df.to_csv(native_csv, index=False)
pd.DataFrame(schema_rows).to_csv(schema_csv, index=False)

notes = {
    "source": "Hugging Face: UTAustin-AIHealth/MedHallu",
    "total_rows_saved": int(len(medhallu_df)),
    "sgi_ready_rows": int((medhallu_df["has_question"] & medhallu_df["has_context"]).sum()),
    "right_rows": int((medhallu_df["label"] == 0).sum()),
    "hallu_rows": int((medhallu_df["label"] == 1).sum()),
    "fix_applied": True,
    "fix_reason": "Used exact MedHallu column names: Question, Knowledge, Ground Truth, Hallucinated Answer.",
}
with open(notes_json, "w", encoding="utf-8") as f:
    json.dump(notes, f, indent=2, ensure_ascii=False)

print("✅ MedHallu naprawiony")
print("Rows total:", len(medhallu_df))
print("Rows label=0:", int((medhallu_df['label'] == 0).sum()))
print("Rows label=1:", int((medhallu_df['label'] == 1).sum()))
print("SGI-ready:", int((medhallu_df["has_question"] & medhallu_df["has_context"]).sum()))

display(pd.DataFrame(schema_rows))
display(medhallu_df.head(6)[[
    "dataset", "subset", "sample_id", "question", "context", "response",
    "label", "has_question", "has_context", "difficulty", "hallucination_category", "source_config"
]])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ MedHallu naprawiony
Rows total: 20000
Rows label=0: 10000
Rows label=1: 10000
SGI-ready: 20000


,file,rows,columns,has_required_cols
0,/content/drive/MyDrive/Hallucination_Project/d...,9000,Question | Knowledge | Ground Truth | Difficul...,True
1,/content/drive/MyDrive/Hallucination_Project/d...,1000,Question | Knowledge | Ground Truth | Difficul...,True


,dataset,subset,sample_id,question,context,response,label,has_question,has_context,difficulty,hallucination_category,source_config
0,MedHallu,easy,pqa_artificial_train_0_right,Are group 2 innate lymphoid cells ( ILC2s ) in...,['Chronic rhinosinusitis (CRS) is a heterogene...,"As ILC2s are elevated in patients with CRSwNP,...",0,True,True,easy,Misinterpretation of #Question#,pqa_artificial
1,MedHallu,easy,pqa_artificial_train_0_hallu,Are group 2 innate lymphoid cells ( ILC2s ) in...,['Chronic rhinosinusitis (CRS) is a heterogene...,Group 2 innate lymphoid cells (ILC2s) are sign...,1,True,True,easy,Misinterpretation of #Question#,pqa_artificial
2,MedHallu,medium,pqa_artificial_train_1_right,Does vagus nerve contribute to the development...,['Phosphatidylethanolamine N-methyltransferase...,Neuronal signals via the hepatic vagus nerve c...,0,True,True,medium,Misinterpretation of #Question#,pqa_artificial
3,MedHallu,medium,pqa_artificial_train_1_hallu,Does vagus nerve contribute to the development...,['Phosphatidylethanolamine N-methyltransferase...,Capsaicin-induced disruption of hepatic affere...,1,True,True,medium,Misinterpretation of #Question#,pqa_artificial
4,MedHallu,medium,pqa_artificial_train_2_right,Does psammaplin A induce Sirtuin 1-dependent a...,['Psammaplin A (PsA) is a natural product isol...,PsA significantly inhibited MCF-7/adr cells pr...,0,True,True,medium,Misinterpretation of #Question#,pqa_artificial
5,MedHallu,medium,pqa_artificial_train_2_hallu,Does psammaplin A induce Sirtuin 1-dependent a...,['Psammaplin A (PsA) is a natural product isol...,Psammaplin A induces Sirtuin 1-independent aut...,1,True,True,medium,Misinterpretation of #Question#,pqa_artificial


## HalluQuestQA — przetwarzanie

HalluQuestQA ma:
- `question_text`
- `ans1_text`
- `ans2_text`
- adnotacje span-level błędów i etykiety mapujące błędy do odpowiedzi.

Tutaj tworzymy po dwa rekordy na wiersz:
- odpowiedź 1,
- odpowiedź 2.

Binarna etykieta jest tu wyprowadzana pośrednio:
jeśli dana odpowiedź ma przypisany co najmniej jeden span błędu lub span błędu występuje w jej treści, oznaczamy ją jako halucynację.

In [ ]:
from pathlib import Path
from google.colab import drive
import pandas as pd
import numpy as np
import json
import csv
import re
import ast

drive.mount("/content/drive", force_remount=False)

PROJECT_ROOT = Path("/content/drive/MyDrive/Hallucination_Project")
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
BY_DATASET_DIR = PROCESSED_DIR / "by_dataset"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
BY_DATASET_DIR.mkdir(parents=True, exist_ok=True)

def normalize_text(x) -> str:
    if x is None:
        return ""
    try:
        if pd.isna(x):
            return ""
    except Exception:
        pass
    text = str(x).strip()
    text = re.sub(r"\s+", " ", text)
    return text

def to_python_list(x):
    if x is None:
        return []
    try:
        if pd.isna(x):
            return []
    except Exception:
        pass
    if isinstance(x, list):
        return x
    if isinstance(x, tuple):
        return list(x)
    if isinstance(x, str):
        s = x.strip()
        if not s:
            return []
        try:
            val = ast.literal_eval(s)
            if isinstance(val, list):
                return val
            return [val]
        except Exception:
            return [s]
    return [x]

def make_standard_df(rows: list[dict]) -> pd.DataFrame:
    df = pd.DataFrame(rows)
    expected = [
        "dataset", "subset", "sample_id", "question", "context", "response",
        "label", "has_question", "has_context", "task_type", "source_file"
    ]
    for c in expected:
        if c not in df.columns:
            df[c] = np.nan

    ordered = expected + [c for c in df.columns if c not in expected]
    df = df[ordered].copy()

    for c in ["dataset", "subset", "sample_id", "question", "context", "response", "task_type", "source_file"]:
        df[c] = df[c].map(normalize_text)

    df["label"] = pd.to_numeric(df["label"], errors="coerce")
    df["has_question"] = df["has_question"].fillna(False).astype(bool)
    df["has_context"] = df["has_context"].fillna(False).astype(bool)
    return df

def read_halluquestqa_robust(csv_path):
    with open(csv_path, "r", encoding="utf-8", errors="ignore") as f:
        sample = f.read(50000)

    try:
        dialect = csv.Sniffer().sniff(sample, delimiters=",\t;|")
        sep = dialect.delimiter
    except Exception:
        sep = ","

    print(f"Detected separator: {repr(sep)}")

    try:
        return pd.read_csv(
            csv_path,
            sep=sep,
            engine="python",
            quotechar='"',
            escapechar="\\",
        )
    except Exception as e1:
        print("Pierwsza próba nieudana:", e1)

    try:
        return pd.read_csv(
            csv_path,
            sep=None,
            engine="python",
            quotechar='"',
            escapechar="\\",
        )
    except Exception as e2:
        print("Druga próba nieudana:", e2)

    return pd.read_csv(
        csv_path,
        sep=sep,
        engine="python",
        quotechar='"',
        escapechar="\\",
        on_bad_lines="warn",
    )

root_candidates = [RAW_DIR / "HalluQuestQA", RAW_DIR / "acl2025-lfqa-hallucination"]
root_candidates = [p for p in root_candidates if p.exists()]

if not root_candidates:
    raise FileNotFoundError("Nie znaleziono folderu HalluQuestQA / acl2025-lfqa-hallucination")

csv_path = None
for root in root_candidates:
    match = next(iter(root.rglob("HaluQuestQA.csv")), None)
    if match is not None:
        csv_path = match
        break

if csv_path is None:
    raise FileNotFoundError("Nie znaleziono pliku HaluQuestQA.csv")

df = read_halluquestqa_robust(csv_path)

print("Plik:", csv_path)
print("Shape:", df.shape)
print("Kolumny:", list(df.columns))

def answer_has_error(row, answer_text: str, answer_token: str) -> int:
    answer_text = normalize_text(answer_text)

    label_cols = [
        "factuality_label",
        "irrelevance_label",
        "incomplete_ans_label",
        "reference_example_label",
    ]
    for col in label_cols:
        if col in row.index:
            vals = [str(v).lower() for v in to_python_list(row[col])]
            if answer_token.lower() in vals:
                return 1

    span_cols = [
        "factuality_span",
        "irrelevance_span",
        "incomplete_ans_span",
        "ques_misconception_span",
        "reference_example_span",
    ]
    for col in span_cols:
        if col in row.index:
            spans = [normalize_text(v) for v in to_python_list(row[col])]
            for sp in spans:
                if sp and sp in answer_text:
                    return 1

    return 0

rows = []
id_col = "source_file" if "source_file" in df.columns else None

for i, row in df.iterrows():
    sid = normalize_text(row[id_col]) if id_col else f"halluquestqa_{i}"
    q = normalize_text(row["question_text"]) if "question_text" in df.columns else ""

    ans1 = normalize_text(row["ans1_text"]) if "ans1_text" in df.columns else ""
    ans2 = normalize_text(row["ans2_text"]) if "ans2_text" in df.columns else ""

    if ans1:
        rows.append({
            "dataset": "HalluQuestQA",
            "subset": normalize_text(row["category"]) if "category" in df.columns else "default",
            "sample_id": f"{sid}_ans1",
            "question": q,
            "context": "",
            "response": ans1,
            "label": answer_has_error(row, ans1, "answer1"),
            "has_question": bool(q),
            "has_context": False,
            "task_type": "lfqa_answer_quality",
            "source_file": str(csv_path),
            "answer_origin": normalize_text(row["ans1_label"]) if "ans1_label" in df.columns else "",
        })

    if ans2:
        rows.append({
            "dataset": "HalluQuestQA",
            "subset": normalize_text(row["category"]) if "category" in df.columns else "default",
            "sample_id": f"{sid}_ans2",
            "question": q,
            "context": "",
            "response": ans2,
            "label": answer_has_error(row, ans2, "answer2"),
            "has_question": bool(q),
            "has_context": False,
            "task_type": "lfqa_answer_quality",
            "source_file": str(csv_path),
            "answer_origin": normalize_text(row["ans2_label"]) if "ans2_label" in df.columns else "",
        })

halluquestqa_df = make_standard_df(rows)

out_dir = BY_DATASET_DIR / "HalluQuestQA"
out_dir.mkdir(parents=True, exist_ok=True)

native_parquet = out_dir / "native.parquet"
native_csv = out_dir / "native.csv"
notes_json = out_dir / "notes.json"

halluquestqa_df.to_parquet(native_parquet, index=False)
halluquestqa_df.to_csv(native_csv, index=False)

notes = {
    "source_file": str(csv_path),
    "total_rows_saved": int(len(halluquestqa_df)),
    "warning": "Binary label inferowany ze spanów i answer-level mappings.",
}
with open(notes_json, "w", encoding="utf-8") as f:
    json.dump(notes, f, indent=2, ensure_ascii=False)

print("✅ HalluQuestQA przetworzony.")
display(halluquestqa_df.head(10))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Detected separator: '\t'
Plik: /content/drive/MyDrive/Hallucination_Project/data/raw/HalluQuestQA/src/data/HaluQuestQA.csv
Shape: (698, 43)
Kolumny: ['Unnamed: 0', 'source_file', 'annotator', 'question_text', 'ans1_text', 'ans2_text', 'factuality_span', 'factuality_reason', 'irrelevance_span', 'irrelevance_reason', 'incomplete_ans_span', 'incomplete_ans_reason', 'ques_misconception_span', 'ques_misconception_reason', 'ans_preference', 'ans_preference_reason', 'ans1_label', 'ans2_label', 'ques_start', 'ans_start', 'reference_example_span', 'reference_example_reason', 'reference_example_helpful', 'ques_misconception_label', 'factuality_label', 'irrelevance_label', 'incomplete_ans_label', 'reference_example_label', 'category', 'factuality_model_score', 'factuality_human_score', 'relevance_model_score', 'relevance_human_score', 'completeness_model_score', 'comple

,dataset,subset,sample_id,question,context,response,label,has_question,has_context,task_type,source_file,answer_origin
0,HalluQuestQA,technology,f0c498d1-45ef-4826-9a78-fe3d95ddacd7.txt_ans1,How does rendering a video game resolution abo...,,It's called supersampling and basically gives ...,1,True,False,lfqa_answer_quality,/content/drive/MyDrive/Hallucination_Project/d...,human_answer
1,HalluQuestQA,technology,f0c498d1-45ef-4826-9a78-fe3d95ddacd7.txt_ans2,How does rendering a video game resolution abo...,,When you render a video game at a resolution h...,1,True,False,lfqa_answer_quality,/content/drive/MyDrive/Hallucination_Project/d...,model_answer
2,HalluQuestQA,technology,2cda5e56-6871-443d-88e0-f98f0d797c1e.txt_ans1,"Why do some download sites have a wait time, f...",,Download sites have a wait time before the dow...,1,True,False,lfqa_answer_quality,/content/drive/MyDrive/Hallucination_Project/d...,model_answer
3,HalluQuestQA,technology,2cda5e56-6871-443d-88e0-f98f0d797c1e.txt_ans2,"Why do some download sites have a wait time, f...",,"So your eyeballs absorb some ads. Or, hopefull...",1,True,False,lfqa_answer_quality,/content/drive/MyDrive/Hallucination_Project/d...,human_answer
4,HalluQuestQA,technology,d8045b30-f9ed-4e74-91db-7d601f2366e4.txt_ans1,How does software that can detect text generat...,,Anomaly detection is the machine learning meth...,1,True,False,lfqa_answer_quality,/content/drive/MyDrive/Hallucination_Project/d...,human_answer
5,HalluQuestQA,technology,d8045b30-f9ed-4e74-91db-7d601f2366e4.txt_ans2,How does software that can detect text generat...,,Chatgpt detection software works by analyzing ...,0,True,False,lfqa_answer_quality,/content/drive/MyDrive/Hallucination_Project/d...,model_answer
6,HalluQuestQA,technology,25a67747-bef1-479a-8c87-869662e1431e.txt_ans1,How is GPS free? GPS has made a major impact o...,,GPS is free because it's a public service prov...,1,True,False,lfqa_answer_quality,/content/drive/MyDrive/Hallucination_Project/d...,model_answer
7,HalluQuestQA,technology,25a67747-bef1-479a-8c87-869662e1431e.txt_ans2,How is GPS free? GPS has made a major impact o...,,"GPS ""receivers"" are just radios that listen to...",1,True,False,lfqa_answer_quality,/content/drive/MyDrive/Hallucination_Project/d...,human_answer
8,HalluQuestQA,technology,96561b7c-cc45-4e9d-9846-e03931774fe8.txt_ans1,How did credit/debit cards work before chip & ...,,"Before chip and pin machines, credit/debit car...",1,True,False,lfqa_answer_quality,/content/drive/MyDrive/Hallucination_Project/d...,model_answer
9,HalluQuestQA,technology,96561b7c-cc45-4e9d-9846-e03931774fe8.txt_ans2,How did credit/debit cards work before chip & ...,,"Originally, the merchant had a small clipboard...",1,True,False,lfqa_answer_quality,/content/drive/MyDrive/Hallucination_Project/d...,human_answer


## Med-MMHL — przetwarzanie

Med-MMHL jest benchmarkiem misinformation/fake-news i ma kilka tasków:
- `fakenews_article`
- `sentence`
- `image_article`
- `fakenews_tweet`
- `image_tweet`

To nie jest klasyczny QA benchmark.
Dlatego ta komórka:
- przeszukuje foldery train/dev/test,
- czyta wszystkie pliki tabelaryczne,
- robi best-effort mapowanie do wspólnego formatu technicznego,
- zapisuje wszystko w jednym folderze datasetowym.

In [ ]:
from pathlib import Path
from google.colab import drive
import pandas as pd
import json
import shutil
from datetime import datetime

drive.mount("/content/drive", force_remount=False)

PROJECT_ROOT = Path("/content/drive/MyDrive/Hallucination_Project")
medmmhl_dir = PROJECT_ROOT / "data" / "processed" / "by_dataset" / "Med-MMHL"

parquet_path = medmmhl_dir / "native.parquet"
csv_path = medmmhl_dir / "native.csv"
notes_path = medmmhl_dir / "notes.json"

if not medmmhl_dir.exists():
    raise FileNotFoundError(f"Brak folderu: {medmmhl_dir}")

if parquet_path.exists():
    df = pd.read_parquet(parquet_path)
    source_used = "parquet"
elif csv_path.exists():
    df = pd.read_csv(csv_path)
    source_used = "csv"
else:
    raise FileNotFoundError("Nie znaleziono native.parquet ani native.csv dla Med-MMHL.")

print("Wczytano:", source_used, "| rows =", len(df))

# backup
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup_dir = medmmhl_dir / f"backup_before_patch_{timestamp}"
backup_dir.mkdir(parents=True, exist_ok=True)

if parquet_path.exists():
    shutil.copy2(parquet_path, backup_dir / "native.parquet")
if csv_path.exists():
    shutil.copy2(csv_path, backup_dir / "native.csv")
if notes_path.exists():
    shutil.copy2(notes_path, backup_dir / "notes.json")

print("Backup:", backup_dir)

# upewnienie się, że kolumny istnieją
for col, default in {
    "question": "",
    "context": "",
    "response": "",
    "has_question": False,
    "has_context": False,
    "task_type": "misinformation_detection",
}.items():
    if col not in df.columns:
        df[col] = default

# normalizacja stringów
for col in ["question", "context", "response", "subset", "task_type"]:
    df[col] = df[col].fillna("").astype(str).str.strip()

# 1) jeśli context == response, to znaczy że fallback zdublował tekst
same_text_mask = (
    df["context"].ne("") &
    df["response"].ne("") &
    (df["context"] == df["response"])
)

# dla takich rekordów zostawiamy tylko response
df.loc[same_text_mask, "context"] = ""
df.loc[same_text_mask, "has_context"] = False

# 2) Med-MMHL nie ma natywnego pytania -> nie udajemy question
df["question"] = ""
df["has_question"] = False

# 3) klasyfikacyjny styl datasetu
df["sgi_setting"] = "not_native_for_sgi"
df["question_mode"] = "none"
df["format_mode"] = "classification_style"

# 4) opcjonalna flaga zgodności SGI po naprawie
df["sgi_compatible_after_patch"] = (
    df["question"].str.strip().ne("") &
    df["context"].str.strip().ne("") &
    df["response"].str.strip().ne("") &
    df["label"].notna()
)

# zapis
df.to_parquet(parquet_path, index=False)
df.to_csv(csv_path, index=False)

notes = {}
if notes_path.exists():
    try:
        with open(notes_path, "r", encoding="utf-8") as f:
            notes = json.load(f)
    except Exception:
        notes = {}

notes["med_mmhl_patch_applied"] = True
notes["patch_timestamp"] = timestamp
notes["format_mode"] = "classification_style"
notes["sgi_setting"] = "not_native_for_sgi"
notes["warning"] = (
    "Med-MMHL is a misinformation detection benchmark, not a native question-context-response dataset. "
    "Duplicated context/response values were cleaned."
)

with open(notes_path, "w", encoding="utf-8") as f:
    json.dump(notes, f, indent=2, ensure_ascii=False)

print("\n✅ Patch zapisany")
print("Duplikaty naprawione:", int(same_text_mask.sum()))

display(df.head(10)[[
    "dataset", "subset", "sample_id", "question", "context", "response",
    "label", "has_question", "has_context", "task_type", "format_mode", "sgi_setting"
]])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Wczytano: parquet | rows = 105215
Backup: /content/drive/MyDrive/Hallucination_Project/data/processed/by_dataset/Med-MMHL/backup_before_patch_20260321_175527

✅ Patch zapisany
Duplikaty naprawione: 75877


,dataset,subset,sample_id,question,context,response,label,has_question,has_context,task_type,format_mode,sgi_setting
0,Med-MMHL,sentence,dev_0,,,No support for the development of the experime...,1,False,False,sentence,classification_style,not_native_for_sgi
1,Med-MMHL,sentence,dev_1,,,Ovarian and endometrial cancer are among the m...,0,False,False,sentence,classification_style,not_native_for_sgi
2,Med-MMHL,sentence,dev_2,,,“Refusing to allow a child to undergo a social...,0,False,False,sentence,classification_style,not_native_for_sgi
3,Med-MMHL,sentence,dev_3,,,• Experts say that following safety tips and p...,0,False,False,sentence,classification_style,not_native_for_sgi
4,Med-MMHL,sentence,dev_4,,,"""We saw a distinct mutational signature caused...",0,False,False,sentence,classification_style,not_native_for_sgi
5,Med-MMHL,sentence,dev_5,,,The opposite opinion is that pregnant people s...,1,False,False,sentence,classification_style,not_native_for_sgi
6,Med-MMHL,sentence,dev_6,,,Recommendations range from topics such as pres...,0,False,False,sentence,classification_style,not_native_for_sgi
7,Med-MMHL,sentence,dev_7,,,"• Performance characteristics (e.g., sensitivi...",0,False,False,sentence,classification_style,not_native_for_sgi
8,Med-MMHL,sentence,dev_8,,,Spending more leisure time sitting is not asso...,1,False,False,sentence,classification_style,not_native_for_sgi
9,Med-MMHL,sentence,dev_9,,,Cancer cells cannot use lipid molecules as fue...,1,False,False,sentence,classification_style,not_native_for_sgi


## Kontrola końcowa po przetworzeniu wszystkich datasetów

Ta komórka:
- sprawdza, które datasety zostały zapisane,
- pokazuje liczbę rekordów,
- pokazuje, ile rekordów ma realnie dostępne:
  - pytanie,
  - kontekst,
  - etykietę,
- pozwala od razu zobaczyć, które datasety nadają się bezpośrednio do dalszych eksperymentów SGI, a które wymagają osobnej adaptacji metodologicznej.

In [ ]:
summary_rows = []

for ds_dir in sorted(BY_DATASET_DIR.iterdir()):
    if not ds_dir.is_dir():
        continue
    native_path = ds_dir / "native.parquet"
    if not native_path.exists():
        continue

    df = pd.read_parquet(native_path)

    summary_rows.append({
        "dataset": ds_dir.name,
        "rows": len(df),
        "with_label": int(df["label"].notna().sum()) if "label" in df.columns else 0,
        "with_question": int(df["has_question"].sum()) if "has_question" in df.columns else 0,
        "with_context": int(df["has_context"].sum()) if "has_context" in df.columns else 0,
        "with_question_and_context": int((df["has_question"] & df["has_context"]).sum()) if {"has_question", "has_context"}.issubset(df.columns) else 0,
        "path": str(native_path),
    })

summary_df = pd.DataFrame(summary_rows).sort_values("dataset").reset_index(drop=True)
display(summary_df)

summary_df.to_csv(BY_DATASET_DIR / "dataset_processing_summary.csv", index=False)
print("✅ Zapisano:", BY_DATASET_DIR / "dataset_processing_summary.csv")

,dataset,rows,with_label,with_question,with_context,with_question_and_context,path
0,HalluQuestQA,1396,1396,1396,0,0,/content/drive/MyDrive/Hallucination_Project/d...
1,HaluEval,64507,64507,64507,60000,60000,/content/drive/MyDrive/Hallucination_Project/d...
2,Med-HALT,59254,0,59254,39590,39590,/content/drive/MyDrive/Hallucination_Project/d...
3,Med-MMHL,105215,105215,0,0,0,/content/drive/MyDrive/Hallucination_Project/d...
4,MedHal,532558,532558,532558,532558,532558,/content/drive/MyDrive/Hallucination_Project/d...
5,MedHallu,20000,20000,20000,20000,20000,/content/drive/MyDrive/Hallucination_Project/d...


✅ Zapisano: /content/drive/MyDrive/Hallucination_Project/data/processed/by_dataset/dataset_processing_summary.csv


# Inicjalizacja modelu

Zrefaktoryzowana sekcja embeddingów: jedna ścieżka uruchomienia dla wszystkich datasetów SGI, z cache na dysku i trybem sharded tylko tam, gdzie naprawdę jest potrzebny.

## Wspólny bootstrap, helpery i cache embeddingów

Ta komórka definiuje model, mean pooling, normalizację L2 oraz jeden wspólny pipeline `run_embedding_pipeline(...)`.

- **HaluEval** i **MedHallu** działają w trybie `flat` (pojedynczy cache).
- **MedHal** działa przez **ten sam pipeline**, ale wewnętrznie w trybie `sharded`, żeby nie zapychać RAM-u na darmowym Colabie.
- Każdy cache ma walidację po `source_signature`, więc po ponownym uruchomieniu sekcji nie liczy się od nowa, jeśli dane wejściowe się nie zmieniły.

In [ ]:
import gc
import json
import random
import shutil
import subprocess
import sys
from pathlib import Path
from typing import Dict, Iterable, Optional

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U", "transformers", "sentence-transformers", "pyarrow"],
    check=True,
)

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from google.colab import drive
from IPython.display import display
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer

# =========================================================
# BOOTSTRAP
# =========================================================
drive.mount("/content/drive", force_remount=False)

PROJECT_ROOT = Path("/content/drive/MyDrive/Hallucination_Project")
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
BY_DATASET_DIR = PROCESSED_DIR / "by_dataset"
EMB_DIR = PROJECT_ROOT / "embeddings"

for path in [RAW_DIR, PROCESSED_DIR, BY_DATASET_DIR, EMB_DIR]:
    path.mkdir(parents=True, exist_ok=True)

# =========================================================
# KONFIG
# =========================================================
# Zachowuję domyślnie mpnet, żeby nie zmieniać logiki eksperymentu.
# Na darmowym Colabie możesz przełączyć na MiniLM, jeśli chcesz szybciej kosztem innych embeddingów.
MODEL_NAME = "sentence-transformers/all-mpnet-base-v2"
# MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

MAX_LENGTH = 384 if "mpnet" in MODEL_NAME.lower() else 256
BATCH_SIZE = 16 if "mpnet" in MODEL_NAME.lower() else 64
CACHE_VERSION = "manual_meanpool_l2_refactor_v1"
TEXT_FIELDS = ["question", "context", "response"]
USECOLS = [
    "dataset", "subset", "sample_id", "question", "context", "response", "label",
    "question_mode", "sgi_setting"
]
META_BASE_COLUMNS = ["dataset", "subset", "sample_id", "label", "question_mode", "sgi_setting"]
TARGET_DATASETS = ["HaluEval", "MedHal", "MedHallu"]

DATASET_RUN_CONFIG = {
    "HaluEval": {
        "storage_mode": "flat",
    },
    "MedHal": {
        "storage_mode": "sharded",
        "csv_chunk_size": 20000,
        "shard_size": 10000,
        "max_shards_this_run": None,
    },
    "MedHallu": {
        "storage_mode": "flat",
    },
}

# =========================================================
# SEED + DEVICE
# =========================================================
def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(42)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE != "cuda":
    raise RuntimeError("Brak GPU. Ustaw runtime z GPU.")

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

print("✅ DEVICE:", DEVICE)
print("✅ GPU:", torch.cuda.get_device_name(0))
print("✅ MODEL_NAME:", MODEL_NAME)
print("✅ MAX_LENGTH:", MAX_LENGTH)
print("✅ BATCH_SIZE:", BATCH_SIZE)
print("✅ CACHE_VERSION:", CACHE_VERSION)

# =========================================================
# MODEL
# =========================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

# =========================================================
# HELPERY OGÓLNE
# =========================================================
def safe_name(text: str) -> str:
    return "".join(ch if ch.isalnum() or ch in "-_." else "_" for ch in text)


def normalize_text(x) -> str:
    if x is None:
        return ""
    try:
        if pd.isna(x):
            return ""
    except Exception:
        pass
    return " ".join(str(x).strip().split())


def mean_pooling(last_hidden_state: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts


def encode_texts_meanpool_l2(
    texts: Iterable[str],
    batch_size: Optional[int] = None,
    max_length: Optional[int] = None,
    desc: str = "Encoding",
) -> np.ndarray:
    texts = list(texts)
    if len(texts) == 0:
        return np.empty((0, model.config.hidden_size), dtype=np.float32)

    batch_size = batch_size or BATCH_SIZE
    max_length = max_length or MAX_LENGTH
    parts = []

    for start in tqdm(range(0, len(texts), batch_size), desc=desc, leave=False, dynamic_ncols=True):
        batch = [normalize_text(x) for x in texts[start:start + batch_size]]
        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )
        encoded = {k: v.to(DEVICE) for k, v in encoded.items()}

        with torch.inference_mode():
            outputs = model(**encoded)
            pooled = mean_pooling(outputs.last_hidden_state, encoded["attention_mask"])
            pooled = F.normalize(pooled, p=2, dim=1)

        parts.append(pooled.detach().cpu().numpy().astype(np.float32))

        del encoded, outputs, pooled
        gc.collect()
        torch.cuda.empty_cache()

    return np.vstack(parts)


def filter_complete_triplets(df: pd.DataFrame) -> pd.DataFrame:
    work = df.copy()

    for col in TEXT_FIELDS:
        if col not in work.columns:
            work[col] = ""
        work[col] = work[col].map(normalize_text)

    if "label" not in work.columns:
        work["label"] = np.nan

    mask = (
        work["question"].astype(str).str.strip().ne("")
        & work["context"].astype(str).str.strip().ne("")
        & work["response"].astype(str).str.strip().ne("")
        & work["label"].notna()
    )

    return work.loc[mask].reset_index(drop=True)


def get_dataset_paths(dataset_name: str) -> Dict[str, Path]:
    ds_dir = BY_DATASET_DIR / dataset_name
    return {
        "dir": ds_dir,
        "csv": ds_dir / "native.csv",
        "parquet": ds_dir / "native.parquet",
        "notes": ds_dir / "notes.json",
    }


def resolve_input_path(dataset_name: str, preferred: str = "parquet") -> Path:
    paths = get_dataset_paths(dataset_name)

    if preferred == "parquet":
        candidates = [paths["parquet"], paths["csv"]]
    elif preferred == "csv":
        candidates = [paths["csv"], paths["parquet"]]
    else:
        raise ValueError("preferred musi być jednym z: 'parquet' / 'csv'")

    for path in candidates:
        if path.exists():
            return path

    raise FileNotFoundError(f"Brak native.csv / native.parquet dla datasetu {dataset_name}")


def read_json(path: Path, default=None):
    if default is None:
        default = {}
    if not path.exists():
        return default
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return default


def write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)


def get_file_signature(path: Path) -> dict:
    stat = path.stat()
    return {
        "path": str(path),
        "size_bytes": int(stat.st_size),
        "mtime_ns": int(stat.st_mtime_ns),
    }


def get_embedding_cache_dir(dataset_name: str, variant_name: str = "complete_triplets") -> Path:
    return EMB_DIR / CACHE_VERSION / safe_name(MODEL_NAME) / dataset_name / variant_name


def load_processed_dataset(dataset_name: str, preferred: str = "parquet") -> pd.DataFrame:
    path = resolve_input_path(dataset_name, preferred=preferred)
    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)
    return pd.read_csv(path)


def select_meta_columns(df: pd.DataFrame) -> list[str]:
    keep = META_BASE_COLUMNS + [c for c in TEXT_FIELDS if c in df.columns]
    return [c for c in keep if c in df.columns]


def encode_single_field(series: pd.Series, field_name: str, dataset_name: str) -> np.ndarray:
    texts = series.astype(str).tolist()

    if field_name == "question" and len(texts) > 0 and series.astype(str).nunique(dropna=False) == 1:
        shared = encode_texts_meanpool_l2(
            [texts[0]],
            desc=f"{dataset_name} | {field_name} (shared)",
        )
        return np.repeat(shared, repeats=len(texts), axis=0)

    return encode_texts_meanpool_l2(
        texts,
        desc=f"{dataset_name} | {field_name}",
    )


def flat_cache_is_valid(out_dir: Path, expected_rows: int, source_signature: dict) -> bool:
    config_path = out_dir / "config.json"
    meta_path = out_dir / "metadata.parquet"

    if not config_path.exists() or not meta_path.exists():
        return False

    config = read_json(config_path, default={})
    if config.get("storage_mode") != "flat":
        return False
    if config.get("source_signature") != source_signature:
        return False

    for field in TEXT_FIELDS:
        if not (out_dir / f"{field}.npy").exists():
            return False

    try:
        meta_rows = len(pd.read_parquet(meta_path, columns=["label"]))
    except Exception:
        meta_rows = len(pd.read_parquet(meta_path))

    return meta_rows == expected_rows


def save_embedding_artifacts_flat(
    df: pd.DataFrame,
    dataset_name: str,
    variant_name: str = "complete_triplets",
    overwrite: bool = False,
    source_signature: Optional[dict] = None,
) -> dict:
    out_dir = get_embedding_cache_dir(dataset_name, variant_name)
    out_dir.mkdir(parents=True, exist_ok=True)

    if source_signature is None:
        source_signature = {}

    if flat_cache_is_valid(out_dir, expected_rows=len(df), source_signature=source_signature) and not overwrite:
        print(f"📦 Cache hit: {dataset_name} | {variant_name}")
        print("   ", out_dir)
        return {
            "dataset_name": dataset_name,
            "storage_mode": "flat",
            "status": "cache_hit",
            "rows": int(len(df)),
            "out_dir": str(out_dir),
        }

    meta_cols = select_meta_columns(df)
    meta_df = df[meta_cols].copy()
    meta_df.to_parquet(out_dir / "metadata.parquet", index=False)

    config = {
        "dataset_name": dataset_name,
        "variant_name": variant_name,
        "storage_mode": "flat",
        "model_name": MODEL_NAME,
        "max_length": MAX_LENGTH,
        "batch_size": BATCH_SIZE,
        "pooling": "manual_mean_pooling",
        "normalization": "l2",
        "rows": int(len(df)),
        "fields": TEXT_FIELDS,
        "source_signature": source_signature,
    }
    write_json(out_dir / "config.json", config)

    for field in TEXT_FIELDS:
        print(f"\n🔹 {dataset_name} | field={field} | rows={len(df)}")
        emb = encode_single_field(df[field], field_name=field, dataset_name=dataset_name)
        np.save(out_dir / f"{field}.npy", emb)
        print(f"   saved: {out_dir / f'{field}.npy'}")
        print(f"   shape: {emb.shape}")

        del emb
        gc.collect()
        torch.cuda.empty_cache()

    print(f"\n✅ Zapisano cache embeddingów: {out_dir}")
    return {
        "dataset_name": dataset_name,
        "storage_mode": "flat",
        "status": "saved",
        "rows": int(len(df)),
        "out_dir": str(out_dir),
    }


def iter_filtered_csv_chunks(csv_path: Path, chunksize: int) -> Iterable[pd.DataFrame]:
    for chunk in pd.read_csv(csv_path, usecols=lambda c: c in USECOLS, chunksize=chunksize):
        filtered = filter_complete_triplets(chunk)
        del chunk
        gc.collect()
        if len(filtered) == 0:
            continue
        yield filtered


def build_manifest_for_sharded_dataset(
    dataset_name: str,
    csv_path: Path,
    out_dir: Path,
    source_signature: dict,
    shard_size: int,
    csv_chunk_size: int,
    overwrite: bool = False,
) -> pd.DataFrame:
    manifest_path = out_dir / "manifest.csv"
    dataset_config_path = out_dir / "dataset_config.json"

    if not overwrite and manifest_path.exists() and dataset_config_path.exists():
        old_cfg = read_json(dataset_config_path, default={})
        if (
            old_cfg.get("storage_mode") == "sharded"
            and old_cfg.get("source_signature") == source_signature
            and int(old_cfg.get("shard_size", -1)) == int(shard_size)
            and int(old_cfg.get("csv_chunk_size", -1)) == int(csv_chunk_size)
        ):
            return pd.read_csv(manifest_path)

    valid_rows_total = 0
    for filtered in tqdm(
        iter_filtered_csv_chunks(csv_path, chunksize=csv_chunk_size),
        desc=f"Scan {dataset_name}",
        leave=True,
        dynamic_ncols=True,
    ):
        valid_rows_total += len(filtered)
        del filtered
        gc.collect()

    n_shards = max(1, (valid_rows_total + shard_size - 1) // shard_size)

    manifest_rows = []
    for shard_id in range(n_shards):
        start_idx = shard_id * shard_size
        end_idx = min((shard_id + 1) * shard_size, valid_rows_total)
        manifest_rows.append(
            {
                "shard_id": shard_id,
                "start_row": start_idx,
                "end_row_exclusive": end_idx,
                "n_rows": end_idx - start_idx,
            }
        )

    manifest_df = pd.DataFrame(manifest_rows)
    manifest_df.to_csv(manifest_path, index=False)

    dataset_config = {
        "dataset_name": dataset_name,
        "storage_mode": "sharded",
        "model_name": MODEL_NAME,
        "max_length": MAX_LENGTH,
        "batch_size": BATCH_SIZE,
        "pooling": "manual_mean_pooling",
        "normalization": "l2",
        "rows": int(valid_rows_total),
        "fields": TEXT_FIELDS,
        "shard_size": int(shard_size),
        "csv_chunk_size": int(csv_chunk_size),
        "source_signature": source_signature,
    }
    write_json(dataset_config_path, dataset_config)

    print("✅ Manifest zapisany:", manifest_path)
    print("valid_rows_total =", valid_rows_total)
    print("n_shards =", n_shards)
    return manifest_df


def collect_shard_df_from_csv(
    csv_path: Path,
    start_row: int,
    end_row: int,
    csv_chunk_size: int,
) -> pd.DataFrame:
    collected_parts = []
    valid_seen = 0

    for filtered in tqdm(
        iter_filtered_csv_chunks(csv_path, chunksize=csv_chunk_size),
        desc=f"Collect rows {start_row}:{end_row}",
        leave=False,
        dynamic_ncols=True,
    ):
        n = len(filtered)
        local_start = valid_seen
        local_end = valid_seen + n

        overlap_start = max(start_row, local_start)
        overlap_end = min(end_row, local_end)

        if overlap_start < overlap_end:
            rel_start = overlap_start - local_start
            rel_end = overlap_end - local_start
            collected_parts.append(filtered.iloc[rel_start:rel_end].copy())

        valid_seen = local_end
        del filtered
        gc.collect()

        if valid_seen >= end_row:
            break

    if not collected_parts:
        raise ValueError(f"Nie udało się zebrać danych dla zakresu {start_row}:{end_row}")

    shard_df = pd.concat(collected_parts, ignore_index=True)
    del collected_parts
    gc.collect()
    return shard_df


def shard_is_complete(shard_dir: Path, expected_rows: int) -> bool:
    if not (shard_dir / "done.json").exists():
        return False
    if not (shard_dir / "metadata.parquet").exists():
        return False
    if not (shard_dir / "config.json").exists():
        return False

    for field in TEXT_FIELDS:
        if not (shard_dir / f"{field}.npy").exists():
            return False

    done = read_json(shard_dir / "done.json", default={})
    return int(done.get("rows", -1)) == int(expected_rows)


def save_embedding_artifacts_sharded(
    dataset_name: str,
    variant_name: str = "complete_triplets",
    overwrite: bool = False,
    shard_size: Optional[int] = None,
    csv_chunk_size: Optional[int] = None,
    max_shards_this_run: Optional[int] = None,
) -> dict:
    ds_cfg = DATASET_RUN_CONFIG.get(dataset_name, {})
    shard_size = int(shard_size or ds_cfg.get("shard_size", 10000))
    csv_chunk_size = int(csv_chunk_size or ds_cfg.get("csv_chunk_size", 20000))
    if max_shards_this_run is None:
        max_shards_this_run = ds_cfg.get("max_shards_this_run", None)

    csv_path = resolve_input_path(dataset_name, preferred="csv")
    source_signature = get_file_signature(csv_path)
    out_dir = get_embedding_cache_dir(dataset_name, variant_name)

    if overwrite and out_dir.exists():
        shutil.rmtree(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    manifest_df = build_manifest_for_sharded_dataset(
        dataset_name=dataset_name,
        csv_path=csv_path,
        out_dir=out_dir,
        source_signature=source_signature,
        shard_size=shard_size,
        csv_chunk_size=csv_chunk_size,
        overwrite=overwrite,
    )

    pending = []
    for row in manifest_df.itertuples(index=False):
        shard_id = int(row.shard_id)
        shard_dir = out_dir / "shards" / f"shard_{shard_id:04d}"
        if not shard_is_complete(shard_dir, expected_rows=int(row.n_rows)):
            pending.append(
                {
                    "shard_id": shard_id,
                    "start_row": int(row.start_row),
                    "end_row_exclusive": int(row.end_row_exclusive),
                    "n_rows": int(row.n_rows),
                }
            )

    if max_shards_this_run is not None:
        pending = pending[: int(max_shards_this_run)]

    print("Pending shards in this run:", len(pending))

    for item in pending:
        shard_id = item["shard_id"]
        start_row = item["start_row"]
        end_row = item["end_row_exclusive"]
        expected_rows = item["n_rows"]

        shard_dir = out_dir / "shards" / f"shard_{shard_id:04d}"
        shard_dir.mkdir(parents=True, exist_ok=True)

        print("\n" + "=" * 100)
        print(f"START SHARD {shard_id} | rows {start_row}:{end_row}")
        print("=" * 100)

        shard_df = collect_shard_df_from_csv(
            csv_path=csv_path,
            start_row=start_row,
            end_row=end_row,
            csv_chunk_size=csv_chunk_size,
        )

        if len(shard_df) != expected_rows:
            print("⚠️ Ostrzeżenie: liczba wierszy shardu różni się od manifestu")
            print("manifest rows:", expected_rows)
            print("actual rows  :", len(shard_df))

        meta_cols = select_meta_columns(shard_df)
        meta_df = shard_df[meta_cols].copy()
        meta_df["row_idx_in_shard"] = np.arange(len(meta_df))
        meta_df.to_parquet(shard_dir / "metadata.parquet", index=False)

        shard_cfg = {
            "dataset_name": dataset_name,
            "variant_name": variant_name,
            "storage_mode": "sharded",
            "model_name": MODEL_NAME,
            "max_length": MAX_LENGTH,
            "batch_size": BATCH_SIZE,
            "pooling": "manual_mean_pooling",
            "normalization": "l2",
            "shard_id": int(shard_id),
            "rows": int(len(shard_df)),
            "fields": TEXT_FIELDS,
            "source_signature": source_signature,
        }
        write_json(shard_dir / "config.json", shard_cfg)

        for field in TEXT_FIELDS:
            emb = encode_single_field(
                shard_df[field],
                field_name=field,
                dataset_name=f"{dataset_name} | shard {shard_id}",
            )
            np.save(shard_dir / f"{field}.npy", emb)
            print(f"✅ {field}.npy", emb.shape)

            del emb
            gc.collect()
            torch.cuda.empty_cache()

        write_json(
            shard_dir / "done.json",
            {
                "shard_id": int(shard_id),
                "rows": int(len(shard_df)),
                "status": "done",
                "source_signature": source_signature,
            },
        )

        del shard_df, meta_df
        gc.collect()
        torch.cuda.empty_cache()

        print(f"✅ DONE SHARD {shard_id}")

    shard_status = []
    for row in manifest_df.itertuples(index=False):
        shard_id = int(row.shard_id)
        shard_dir = out_dir / "shards" / f"shard_{shard_id:04d}"
        shard_status.append(shard_is_complete(shard_dir, expected_rows=int(row.n_rows)))

    overall_complete = bool(shard_status) and all(shard_status)
    if len(pending) == 0 and overall_complete:
        status = "cache_hit"
    elif overall_complete:
        status = "saved"
    else:
        status = "partial"

    print("\n✅ Run finished.")
    print("Complete shards:", int(sum(shard_status)), "/", len(shard_status))

    return {
        "dataset_name": dataset_name,
        "storage_mode": "sharded",
        "status": status,
        "rows": int(manifest_df["n_rows"].sum()),
        "n_shards": int(len(manifest_df)),
        "completed_shards": int(sum(shard_status)),
        "out_dir": str(out_dir),
    }


def run_embedding_pipeline(
    dataset_name: str,
    variant_name: str = "complete_triplets",
    overwrite: bool = False,
    max_shards_this_run: Optional[int] = None,
) -> dict:
    if dataset_name not in DATASET_RUN_CONFIG:
        raise ValueError(f"Brak konfiguracji dla datasetu: {dataset_name}")

    ds_cfg = DATASET_RUN_CONFIG[dataset_name]
    storage_mode = ds_cfg.get("storage_mode", "flat")

    print("\n" + "#" * 110)
    print(f"DATASET: {dataset_name} | storage_mode={storage_mode}")
    print("#" * 110)

    if storage_mode == "flat":
        source_path = resolve_input_path(dataset_name, preferred="parquet")
        source_signature = get_file_signature(source_path)
        df = load_processed_dataset(dataset_name, preferred="parquet")
        df_ready = filter_complete_triplets(df)

        print("Oryginalny rozmiar:", len(df))
        print("Po filtrze complete_triplets:", len(df_ready))

        if len(df_ready) == 0:
            raise ValueError(f"Brak kompletnych rekordów w {dataset_name}.")

        display(df_ready.head(3)[["dataset", "subset", "sample_id", "question", "context", "response", "label"]])

        result = save_embedding_artifacts_flat(
            df=df_ready,
            dataset_name=dataset_name,
            variant_name=variant_name,
            overwrite=overwrite,
            source_signature=source_signature,
        )
        del df, df_ready
        gc.collect()
        torch.cuda.empty_cache()
        return result

    if storage_mode == "sharded":
        return save_embedding_artifacts_sharded(
            dataset_name=dataset_name,
            variant_name=variant_name,
            overwrite=overwrite,
            shard_size=ds_cfg.get("shard_size", 10000),
            csv_chunk_size=ds_cfg.get("csv_chunk_size", 20000),
            max_shards_this_run=max_shards_this_run,
        )

    raise ValueError(f"Nieobsługiwany storage_mode={storage_mode}")


def get_embedding_status(dataset_names: Optional[list[str]] = None, variant_name: str = "complete_triplets") -> pd.DataFrame:
    dataset_names = dataset_names or TARGET_DATASETS
    rows = []

    for dataset_name in dataset_names:
        out_dir = get_embedding_cache_dir(dataset_name, variant_name)
        ds_cfg = DATASET_RUN_CONFIG.get(dataset_name, {})
        storage_mode = ds_cfg.get("storage_mode", "unknown")

        row = {
            "dataset": dataset_name,
            "storage_mode": storage_mode,
            "cache_dir": str(out_dir),
            "ready": False,
            "details": "",
        }

        if storage_mode == "flat":
            cfg_path = out_dir / "config.json"
            meta_path = out_dir / "metadata.parquet"
            files_ok = cfg_path.exists() and meta_path.exists() and all(
                (out_dir / f"{field}.npy").exists() for field in TEXT_FIELDS
            )
            row["ready"] = files_ok
            if cfg_path.exists():
                cfg = read_json(cfg_path, default={})
                row["details"] = f"rows={cfg.get('rows', '?')}"

        elif storage_mode == "sharded":
            manifest_path = out_dir / "manifest.csv"
            if manifest_path.exists():
                manifest_df = pd.read_csv(manifest_path)
                done_count = 0
                for shard_row in manifest_df.itertuples(index=False):
                    shard_dir = out_dir / "shards" / f"shard_{int(shard_row.shard_id):04d}"
                    if shard_is_complete(shard_dir, expected_rows=int(shard_row.n_rows)):
                        done_count += 1
                row["ready"] = done_count == len(manifest_df) and len(manifest_df) > 0
                row["details"] = f"shards={done_count}/{len(manifest_df)}"
            else:
                row["details"] = "manifest missing"

        rows.append(row)

    return pd.DataFrame(rows)


print("\n✅ Model i helpery gotowe.")
display(get_embedding_status())


## Status cache (opcjonalnie)

Ta komórka pokazuje, co już jest policzone. Po pierwszym pełnym uruchomieniu możesz wchodzić w pojedyncze sekcje i notebook nie powinien liczyć tego ponownie, dopóki nie zmienią się pliki `native.csv` / `native.parquet`.

In [ ]:
status_df = get_embedding_status()
display(status_df)


## HaluEval — embeddingi, mean pooling i normalizacja L2

In [ ]:
halueval_result = run_embedding_pipeline("HaluEval")
print("\n📁 HaluEval embeddings zapisane w:")
print(halueval_result["out_dir"])


## MedHal — ten sam pipeline, ale w trybie sharded na dysk

Na poziomie wywołania nic tu nie jest specjalne: nadal odpalasz `run_embedding_pipeline(...)`. Różnica jest tylko wewnątrz helperów, żeby utrzymać zużycie RAM pod kontrolą na darmowym Colabie.

In [ ]:
medhal_result = run_embedding_pipeline("MedHal")
print("\n📁 MedHal embeddings zapisane w:")
print(medhal_result["out_dir"])


## MedHallu — embeddingi, mean pooling i normalizacja L2

In [ ]:
medhallu_result = run_embedding_pipeline("MedHallu")
print("\n📁 MedHallu embeddings zapisane w:")
print(medhallu_result["out_dir"])


## Podsumowanie stanu po uruchomieniu sekcji embeddingowych

In [ ]:
status_df = get_embedding_status()
display(status_df)


# Komórka 5: Obliczanie kątów i SGI (Semantic Grounding Index)

W tej sekcji liczymy odległości kątowe i wskaźnik SGI bez ponownego liczenia embeddingów.

Założenia:
- embeddingi `question`, `context`, `response` są już zapisane na dysku po sekcji inicjalizacji modelu,
- kąty liczymy jako `theta = arccos(clip(cosine, -1, 1))`,
- wskaźnik liczymy jako `SGI(r; q, c) = theta(r, q) / (theta(r, c) + epsilon)`,
- wynik zapisujemy do **CSV** jako główną tabelę z nowymi kolumnami:
  - `theta_rq`
  - `theta_rc`
  - `sgi`

Implementacja działa zarówno dla:
- `flat` — HaluEval, MedHallu
- `sharded` — MedHal

Dzięki temu można uruchamiać tę sekcję niezależnie, o ile cache embeddingów został policzony wcześniej i jest zapisany na dysku.


In [ ]:

import gc
from pathlib import Path

import numpy as np
import pandas as pd

SGI_EPSILON = 1e-8
SGI_CHUNK_SIZE = 50_000
SGI_DIR = PROJECT_ROOT / "sgi" / CACHE_VERSION / safe_name(MODEL_NAME)


def get_sgi_output_dir(dataset_name: str, variant_name: str = "complete_triplets") -> Path:
    return SGI_DIR / dataset_name / variant_name


def rowwise_angular_distance(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    if a.shape != b.shape:
        raise ValueError(f"Niezgodne kształty embeddingów: {a.shape} vs {b.shape}")

    cosine = np.sum(a * b, axis=1, dtype=np.float32)
    cosine = np.clip(cosine, -1.0, 1.0)
    theta = np.arccos(cosine).astype(np.float32)
    return theta


def compute_sgi_from_triplet_embeddings(
    q_emb: np.ndarray,
    c_emb: np.ndarray,
    r_emb: np.ndarray,
    epsilon: float = SGI_EPSILON,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    theta_rq = rowwise_angular_distance(r_emb, q_emb)
    theta_rc = rowwise_angular_distance(r_emb, c_emb)
    sgi = (theta_rq / (theta_rc + float(epsilon))).astype(np.float32)
    return theta_rq, theta_rc, sgi


def load_memmap_embeddings(base_dir: Path) -> dict:
    payload = {}
    for field in TEXT_FIELDS:
        npy_path = base_dir / f"{field}.npy"
        if not npy_path.exists():
            raise FileNotFoundError(f"Brak pliku embeddingów: {npy_path}")
        payload[field] = np.load(npy_path, mmap_mode="r")
    return payload


def save_sgi_chunk_to_csv(
    meta_chunk: pd.DataFrame,
    theta_rq: np.ndarray,
    theta_rc: np.ndarray,
    sgi: np.ndarray,
    csv_path: Path,
    write_header: bool,
) -> None:
    out_chunk = meta_chunk.copy()
    out_chunk["theta_rq"] = theta_rq
    out_chunk["theta_rc"] = theta_rc
    out_chunk["sgi"] = sgi
    out_chunk.to_csv(csv_path, mode="w" if write_header else "a", header=write_header, index=False)


def write_sgi_config(config_path: Path, payload: dict) -> None:
    config_path.parent.mkdir(parents=True, exist_ok=True)
    write_json(config_path, payload)


def sgi_cache_exists(out_dir: Path) -> bool:
    csv_path = out_dir / "sgi_table.csv"
    cfg_path = out_dir / "config.json"
    return csv_path.exists() and cfg_path.exists() and csv_path.stat().st_size > 0


def compute_sgi_for_flat_cache(
    dataset_name: str,
    variant_name: str = "complete_triplets",
    overwrite: bool = False,
    chunk_size: int = SGI_CHUNK_SIZE,
    epsilon: float = SGI_EPSILON,
) -> dict:
    emb_dir = get_embedding_cache_dir(dataset_name, variant_name)
    out_dir = get_sgi_output_dir(dataset_name, variant_name)
    out_dir.mkdir(parents=True, exist_ok=True)

    out_csv = out_dir / "sgi_table.csv"
    out_cfg = out_dir / "config.json"

    emb_cfg = read_json(emb_dir / "config.json", default={})
    meta_path = emb_dir / "metadata.parquet"
    if not meta_path.exists():
        raise FileNotFoundError(f"Brak metadata.parquet: {meta_path}")

    if sgi_cache_exists(out_dir) and not overwrite:
        print(f"📦 SGI cache hit: {dataset_name} | {variant_name}")
        return {
            "dataset_name": dataset_name,
            "storage_mode": "flat",
            "status": "cache_hit",
            "rows": int(read_json(out_cfg, default={}).get("rows", emb_cfg.get("rows", -1))),
            "csv_path": str(out_csv),
            "out_dir": str(out_dir),
        }

    meta_df = pd.read_parquet(meta_path)
    emb = load_memmap_embeddings(emb_dir)

    n_rows = len(meta_df)
    if not (len(emb["question"]) == len(emb["context"]) == len(emb["response"]) == n_rows):
        raise ValueError("Niezgodna liczba wierszy między metadata.parquet i plikami .npy")

    if out_csv.exists():
        out_csv.unlink()

    for start in range(0, n_rows, chunk_size):
        end = min(start + chunk_size, n_rows)

        q_chunk = np.asarray(emb["question"][start:end], dtype=np.float32)
        c_chunk = np.asarray(emb["context"][start:end], dtype=np.float32)
        r_chunk = np.asarray(emb["response"][start:end], dtype=np.float32)

        theta_rq, theta_rc, sgi = compute_sgi_from_triplet_embeddings(
            q_emb=q_chunk,
            c_emb=c_chunk,
            r_emb=r_chunk,
            epsilon=epsilon,
        )

        meta_chunk = meta_df.iloc[start:end].reset_index(drop=True)
        save_sgi_chunk_to_csv(
            meta_chunk=meta_chunk,
            theta_rq=theta_rq,
            theta_rc=theta_rc,
            sgi=sgi,
            csv_path=out_csv,
            write_header=(start == 0),
        )

        del q_chunk, c_chunk, r_chunk, theta_rq, theta_rc, sgi, meta_chunk
        gc.collect()

    write_sgi_config(
        out_cfg,
        {
            "dataset_name": dataset_name,
            "variant_name": variant_name,
            "storage_mode": "flat",
            "rows": int(n_rows),
            "epsilon": float(epsilon),
            "chunk_size": int(chunk_size),
            "source_embedding_dir": str(emb_dir),
            "source_signature": emb_cfg.get("source_signature", {}),
            "formula": "SGI(r; q, c) = theta(r, q) / (theta(r, c) + epsilon)",
            "angle_formula": "theta = arccos(clip(cosine, -1, 1))",
        },
    )

    return {
        "dataset_name": dataset_name,
        "storage_mode": "flat",
        "status": "saved",
        "rows": int(n_rows),
        "csv_path": str(out_csv),
        "out_dir": str(out_dir),
    }


def compute_sgi_for_sharded_cache(
    dataset_name: str,
    variant_name: str = "complete_triplets",
    overwrite: bool = False,
    chunk_size: int = SGI_CHUNK_SIZE,
    epsilon: float = SGI_EPSILON,
) -> dict:
    emb_dir = get_embedding_cache_dir(dataset_name, variant_name)
    out_dir = get_sgi_output_dir(dataset_name, variant_name)
    out_dir.mkdir(parents=True, exist_ok=True)

    manifest_path = emb_dir / "manifest.csv"
    dataset_cfg_path = emb_dir / "dataset_config.json"
    if not manifest_path.exists():
        raise FileNotFoundError(f"Brak manifest.csv. Najpierw uruchom embeddingi dla {dataset_name}.")

    out_csv = out_dir / "sgi_table.csv"
    out_cfg = out_dir / "config.json"

    manifest_df = pd.read_csv(manifest_path)
    emb_cfg = read_json(dataset_cfg_path, default={})

    if sgi_cache_exists(out_dir) and not overwrite:
        print(f"📦 SGI cache hit: {dataset_name} | {variant_name}")
        return {
            "dataset_name": dataset_name,
            "storage_mode": "sharded",
            "status": "cache_hit",
            "rows": int(read_json(out_cfg, default={}).get("rows", manifest_df["n_rows"].sum())),
            "csv_path": str(out_csv),
            "out_dir": str(out_dir),
        }

    if out_csv.exists():
        out_csv.unlink()

    total_rows = 0
    first_write = True

    for row in manifest_df.itertuples(index=False):
        shard_id = int(row.shard_id)
        shard_dir = emb_dir / "shards" / f"shard_{shard_id:04d}"

        if not shard_is_complete(shard_dir, expected_rows=int(row.n_rows)):
            raise RuntimeError(
                f"Shard {shard_id} w {dataset_name} nie jest kompletny. "
                "Najpierw dokończ sekcję embeddingów."
            )

        meta_df = pd.read_parquet(shard_dir / "metadata.parquet")
        emb = load_memmap_embeddings(shard_dir)

        shard_rows = len(meta_df)
        if not (len(emb["question"]) == len(emb["context"]) == len(emb["response"]) == shard_rows):
            raise ValueError(f"Niezgodna liczba wierszy w shardzie {shard_id}")

        for start in range(0, shard_rows, chunk_size):
            end = min(start + chunk_size, shard_rows)

            q_chunk = np.asarray(emb["question"][start:end], dtype=np.float32)
            c_chunk = np.asarray(emb["context"][start:end], dtype=np.float32)
            r_chunk = np.asarray(emb["response"][start:end], dtype=np.float32)

            theta_rq, theta_rc, sgi = compute_sgi_from_triplet_embeddings(
                q_emb=q_chunk,
                c_emb=c_chunk,
                r_emb=r_chunk,
                epsilon=epsilon,
            )

            meta_chunk = meta_df.iloc[start:end].reset_index(drop=True)
            save_sgi_chunk_to_csv(
                meta_chunk=meta_chunk,
                theta_rq=theta_rq,
                theta_rc=theta_rc,
                sgi=sgi,
                csv_path=out_csv,
                write_header=first_write,
            )
            first_write = False
            total_rows += len(meta_chunk)

            del q_chunk, c_chunk, r_chunk, theta_rq, theta_rc, sgi, meta_chunk
            gc.collect()

        del meta_df, emb
        gc.collect()

    write_sgi_config(
        out_cfg,
        {
            "dataset_name": dataset_name,
            "variant_name": variant_name,
            "storage_mode": "sharded",
            "rows": int(total_rows),
            "epsilon": float(epsilon),
            "chunk_size": int(chunk_size),
            "source_embedding_dir": str(emb_dir),
            "source_signature": emb_cfg.get("source_signature", {}),
            "formula": "SGI(r; q, c) = theta(r, q) / (theta(r, c) + epsilon)",
            "angle_formula": "theta = arccos(clip(cosine, -1, 1))",
            "n_shards": int(len(manifest_df)),
        },
    )

    return {
        "dataset_name": dataset_name,
        "storage_mode": "sharded",
        "status": "saved",
        "rows": int(total_rows),
        "csv_path": str(out_csv),
        "out_dir": str(out_dir),
    }


def run_sgi_pipeline(
    dataset_name: str,
    variant_name: str = "complete_triplets",
    overwrite: bool = False,
    chunk_size: int = SGI_CHUNK_SIZE,
    epsilon: float = SGI_EPSILON,
) -> dict:
    if dataset_name not in DATASET_RUN_CONFIG:
        raise ValueError(f"Brak konfiguracji dla datasetu: {dataset_name}")

    storage_mode = DATASET_RUN_CONFIG[dataset_name].get("storage_mode", "flat")

    print("\n" + "#" * 110)
    print(f"SGI | DATASET: {dataset_name} | storage_mode={storage_mode}")
    print("#" * 110)

    if storage_mode == "flat":
        return compute_sgi_for_flat_cache(
            dataset_name=dataset_name,
            variant_name=variant_name,
            overwrite=overwrite,
            chunk_size=chunk_size,
            epsilon=epsilon,
        )

    if storage_mode == "sharded":
        return compute_sgi_for_sharded_cache(
            dataset_name=dataset_name,
            variant_name=variant_name,
            overwrite=overwrite,
            chunk_size=chunk_size,
            epsilon=epsilon,
        )

    raise ValueError(f"Nieobsługiwany storage_mode={storage_mode}")


def get_sgi_status(dataset_names: list[str] | None = None, variant_name: str = "complete_triplets") -> pd.DataFrame:
    dataset_names = dataset_names or TARGET_DATASETS
    rows = []

    for dataset_name in dataset_names:
        out_dir = get_sgi_output_dir(dataset_name, variant_name)
        cfg = read_json(out_dir / "config.json", default={})
        csv_path = out_dir / "sgi_table.csv"

        rows.append(
            {
                "dataset": dataset_name,
                "ready": csv_path.exists() and csv_path.stat().st_size > 0,
                "csv_path": str(csv_path),
                "rows": int(cfg.get("rows", 0)) if cfg else 0,
                "storage_mode": cfg.get("storage_mode", DATASET_RUN_CONFIG.get(dataset_name, {}).get("storage_mode", "unknown")),
            }
        )

    return pd.DataFrame(rows)


print("✅ Helpery SGI gotowe.")
display(get_sgi_status())


## Uruchomienie SGI dla wybranych datasetów

In [ ]:
halueval_sgi_result = run_sgi_pipeline("HaluEval")
print("\n📄 HaluEval SGI CSV:")
print(halueval_sgi_result["csv_path"])


In [ ]:
medhal_sgi_result = run_sgi_pipeline("MedHal")
print("\n📄 MedHal SGI CSV:")
print(medhal_sgi_result["csv_path"])


In [ ]:
medhallu_sgi_result = run_sgi_pipeline("MedHallu")
print("\n📄 MedHallu SGI CSV:")
print(medhallu_sgi_result["csv_path"])


## Status plików SGI po uruchomieniu

In [ ]:
display(get_sgi_status())


# Komórka 6: Ewaluacja modelu (ROC-AUC, strojone Accuracy, podział na długości)

Ta sekcja:
- czyta zapisane wcześniej tabele `sgi_table.csv`,
- liczy **ROC-AUC** i **Accuracy dla domyślnego progu SGI = 1.0**,
- znajduje **dokładnie najlepszy próg SGI** maksymalizujący Accuracy,
- dzieli próbki na **short / medium / long** dla pytań i odpowiedzi,
- zapisuje wyniki ewaluacji do osobnych plików CSV.

Domyślnie długości są liczone w **tokenach whitespace**, ale można to przełączyć na liczbę znaków.


In [ ]:

import gc
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import accuracy_score, roc_auc_score

EVAL_DIR = PROJECT_ROOT / "evaluation" / CACHE_VERSION / safe_name(MODEL_NAME)

# =========================================================
# KONFIG EWALUACJI
# =========================================================
LENGTH_MODE = "tokens"      # "tokens" | "chars"
LENGTH_SPLIT_MODE = "tertiles"  # na razie 3 grupy: short / medium / long
EVAL_CHUNK_SIZE = 100_000
DEFAULT_SGI_THRESHOLD = 1.0

LENGTH_LABELS = ["short", "medium", "long"]
PARAMETER_COLUMNS = ["variant_name", "question_mode", "sgi_setting"]

# =========================================================
# HELPERY EWALUACJI
# =========================================================
def get_eval_output_dir(dataset_name: str, variant_name: str = "complete_triplets") -> Path:
    return EVAL_DIR / dataset_name / variant_name


def safe_float(x, default=np.nan):
    try:
        out = float(x)
        if np.isfinite(out):
            return out
    except Exception:
        pass
    return default


def compute_length_series(series: pd.Series, mode: str = LENGTH_MODE) -> pd.Series:
    s = series.fillna("").astype(str)
    if mode == "chars":
        return s.str.len().astype(np.int32)
    if mode == "tokens":
        return (
            s.str.strip()
             .str.split()
             .map(lambda x: len(x) if isinstance(x, list) else 0)
             .astype(np.int32)
        )
    raise ValueError(f"Nieobsługiwany LENGTH_MODE: {mode}")


def infer_usecols_for_eval(csv_path: Path) -> list[str]:
    header = pd.read_csv(csv_path, nrows=0)
    wanted = [
        "dataset", "subset", "sample_id", "label", "sgi",
        "question", "response", "question_mode", "sgi_setting"
    ]
    return [c for c in wanted if c in header.columns]


def build_eval_frame_from_sgi_csv(
    csv_path: Path,
    length_mode: str = LENGTH_MODE,
    chunksize: int = EVAL_CHUNK_SIZE,
) -> pd.DataFrame:
    usecols = infer_usecols_for_eval(csv_path)
    if "label" not in usecols or "sgi" not in usecols:
        raise ValueError(f"Plik {csv_path} nie zawiera wymaganych kolumn label / sgi.")

    parts = []
    for chunk in pd.read_csv(csv_path, usecols=usecols, chunksize=chunksize):
        work = chunk.copy()

        work["label"] = pd.to_numeric(work["label"], errors="coerce")
        work["sgi"] = pd.to_numeric(work["sgi"], errors="coerce")

        if "question" in work.columns:
            work["question_length"] = compute_length_series(work["question"], mode=length_mode)
        else:
            work["question_length"] = 0

        if "response" in work.columns:
            work["response_length"] = compute_length_series(work["response"], mode=length_mode)
        else:
            work["response_length"] = 0

        keep_cols = [
            c for c in [
                "dataset", "subset", "sample_id", "label", "sgi",
                "question_mode", "sgi_setting",
                "question_length", "response_length"
            ] if c in work.columns
        ]
        work = work[keep_cols].copy()

        work = work[
            work["label"].notna()
            & work["sgi"].notna()
            & np.isfinite(work["sgi"].to_numpy(dtype=np.float64, copy=False))
        ].reset_index(drop=True)

        if len(work):
            work["label"] = work["label"].astype(np.int8)
            work["sgi"] = work["sgi"].astype(np.float32)
            parts.append(work)

        del chunk, work
        gc.collect()

    if not parts:
        return pd.DataFrame(columns=[
            "dataset", "subset", "sample_id", "label", "sgi",
            "question_mode", "sgi_setting", "question_length", "response_length"
        ])

    return pd.concat(parts, ignore_index=True)


def assign_tertile_groups(lengths: pd.Series, labels: list[str] = LENGTH_LABELS) -> pd.Series:
    if len(lengths) == 0:
        return pd.Series([], dtype="object")

    ranked = lengths.rank(method="first")
    return pd.qcut(ranked, q=3, labels=labels).astype(str)


def attach_length_groups(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["question_length_group"] = assign_tertile_groups(out["question_length"])
    out["response_length_group"] = assign_tertile_groups(out["response_length"])
    return out


def compute_binary_metrics(y_true: np.ndarray, scores: np.ndarray, default_threshold: float = DEFAULT_SGI_THRESHOLD) -> dict:
    y = np.asarray(y_true, dtype=np.int8)
    s = np.asarray(scores, dtype=np.float64)

    valid = np.isfinite(s)
    y = y[valid]
    s = s[valid]

    n = len(y)
    if n == 0:
        return {
            "n_samples": 0,
            "n_pos": 0,
            "n_neg": 0,
            "positive_rate": np.nan,
            "roc_auc": np.nan,
            "accuracy_at_default_threshold": np.nan,
            "optimal_threshold": np.nan,
            "max_accuracy": np.nan,
        }

    n_pos = int((y == 1).sum())
    n_neg = int((y == 0).sum())
    pos_rate = n_pos / n if n else np.nan

    acc_default = accuracy_score(y, (s >= float(default_threshold)).astype(np.int8))

    if len(np.unique(y)) < 2:
        return {
            "n_samples": int(n),
            "n_pos": n_pos,
            "n_neg": n_neg,
            "positive_rate": pos_rate,
            "roc_auc": np.nan,
            "accuracy_at_default_threshold": float(acc_default),
            "optimal_threshold": np.nan,
            "max_accuracy": np.nan,
        }

    roc_auc = roc_auc_score(y, s)

    order = np.argsort(-s, kind="mergesort")
    s_sorted = s[order]
    y_sorted = y[order]

    total_pos = int((y_sorted == 1).sum())
    total_neg = int((y_sorted == 0).sum())
    n_total = len(y_sorted)

    # stan bazowy: próg > max(score) => wszystko jako klasa 0
    best_acc = total_neg / n_total
    best_thr = np.nextafter(np.max(s_sorted), np.inf)

    tp = 0
    fp = 0
    i = 0
    while i < n_total:
        current_score = s_sorted[i]
        j = i
        block = y_sorted[i:j]
        while j < n_total and s_sorted[j] == current_score:
            if y_sorted[j] == 1:
                tp += 1
            else:
                fp += 1
            j += 1

        tn = total_neg - fp
        acc = (tp + tn) / n_total
        thr = float(current_score)

        better = acc > best_acc + 1e-12
        tie = abs(acc - best_acc) <= 1e-12

        if better or (tie and abs(thr - default_threshold) < abs(best_thr - default_threshold)):
            best_acc = acc
            best_thr = thr

        i = j

    return {
        "n_samples": int(n),
        "n_pos": n_pos,
        "n_neg": n_neg,
        "positive_rate": pos_rate,
        "roc_auc": float(roc_auc),
        "accuracy_at_default_threshold": float(acc_default),
        "optimal_threshold": float(best_thr),
        "max_accuracy": float(best_acc),
    }


def summarize_group_metrics(
    df: pd.DataFrame,
    dataset_name: str,
    variant_name: str,
    model_name: str = MODEL_NAME,
    cache_version: str = CACHE_VERSION,
    question_group: str = "all",
    response_group: str = "all",
    evaluation_scope: str = "overall",
) -> dict:
    metrics = compute_binary_metrics(
        y_true=df["label"].to_numpy(dtype=np.int8, copy=False),
        scores=df["sgi"].to_numpy(dtype=np.float32, copy=False),
        default_threshold=DEFAULT_SGI_THRESHOLD,
    )

    row = {
        "dataset_name": dataset_name,
        "variant_name": variant_name,
        "model_name": model_name,
        "model_slug": safe_name(model_name),
        "cache_version": cache_version,
        "length_mode": LENGTH_MODE,
        "length_split_mode": LENGTH_SPLIT_MODE,
        "default_threshold": float(DEFAULT_SGI_THRESHOLD),
        "evaluation_scope": evaluation_scope,
        "question_length_group": question_group,
        "response_length_group": response_group,
        "question_length_min": safe_float(df["question_length"].min()) if len(df) else np.nan,
        "question_length_max": safe_float(df["question_length"].max()) if len(df) else np.nan,
        "question_length_mean": safe_float(df["question_length"].mean()) if len(df) else np.nan,
        "response_length_min": safe_float(df["response_length"].min()) if len(df) else np.nan,
        "response_length_max": safe_float(df["response_length"].max()) if len(df) else np.nan,
        "response_length_mean": safe_float(df["response_length"].mean()) if len(df) else np.nan,
        "question_mode": "mixed",
        "sgi_setting": "mixed",
        "n_subsets": int(df["subset"].nunique()) if "subset" in df.columns else np.nan,
    }
    row.update(metrics)

    if "question_mode" in df.columns and df["question_mode"].nunique(dropna=False) == 1:
        row["question_mode"] = str(df["question_mode"].iloc[0])

    if "sgi_setting" in df.columns and df["sgi_setting"].nunique(dropna=False) == 1:
        row["sgi_setting"] = str(df["sgi_setting"].iloc[0])

    return row


def evaluate_sgi_csv(
    dataset_name: str,
    variant_name: str = "complete_triplets",
    overwrite: bool = False,
) -> dict:
    sgi_dir = get_sgi_output_dir(dataset_name, variant_name)
    sgi_csv = sgi_dir / "sgi_table.csv"
    sgi_cfg = sgi_dir / "config.json"

    if not sgi_csv.exists():
        raise FileNotFoundError(f"Brak sgi_table.csv dla {dataset_name}: {sgi_csv}")

    out_dir = get_eval_output_dir(dataset_name, variant_name)
    out_dir.mkdir(parents=True, exist_ok=True)

    overall_csv = out_dir / "evaluation_overall.csv"
    length_csv = out_dir / "evaluation_length_splits.csv"
    config_json = out_dir / "config.json"

    if overall_csv.exists() and length_csv.exists() and config_json.exists() and not overwrite:
        print(f"📦 Evaluation cache hit: {dataset_name} | {variant_name}")
        cfg = read_json(config_json, default={})
        return {
            "dataset_name": dataset_name,
            "variant_name": variant_name,
            "status": "cache_hit",
            "overall_csv": str(overall_csv),
            "length_csv": str(length_csv),
            "rows_used": int(cfg.get("rows_used", -1)),
        }

    eval_df = build_eval_frame_from_sgi_csv(sgi_csv, length_mode=LENGTH_MODE, chunksize=EVAL_CHUNK_SIZE)
    if len(eval_df) == 0:
        raise ValueError(f"Brak poprawnych wierszy do ewaluacji w {sgi_csv}")

    eval_df = attach_length_groups(eval_df)

    overall_rows = []
    overall_rows.append(
        summarize_group_metrics(
            eval_df,
            dataset_name=dataset_name,
            variant_name=variant_name,
            evaluation_scope="overall",
        )
    )

    # Dodatkowe przekroje po parametrach, jeśli rzeczywiście występują różne wartości
    param_cols_present = [c for c in ["question_mode", "sgi_setting"] if c in eval_df.columns]
    if param_cols_present and any(eval_df[c].nunique(dropna=False) > 1 for c in param_cols_present):
        group_cols = [c for c in param_cols_present if eval_df[c].nunique(dropna=False) > 1]
        for keys, group_df in eval_df.groupby(group_cols, dropna=False):
            if not isinstance(keys, tuple):
                keys = (keys,)
            row = summarize_group_metrics(
                group_df,
                dataset_name=dataset_name,
                variant_name=variant_name,
                evaluation_scope="parameter_slice",
            )
            for col, val in zip(group_cols, keys):
                row[col] = str(val)
            overall_rows.append(row)

    overall_df = pd.DataFrame(overall_rows)

    length_rows = []
    for (q_group, r_group), group_df in eval_df.groupby(["question_length_group", "response_length_group"], dropna=False):
        if len(group_df) == 0:
            continue
        row = summarize_group_metrics(
            group_df,
            dataset_name=dataset_name,
            variant_name=variant_name,
            question_group=str(q_group),
            response_group=str(r_group),
            evaluation_scope="length_slice",
        )
        row["question_length_group"] = str(q_group)
        row["response_length_group"] = str(r_group)
        length_rows.append(row)

    length_df = pd.DataFrame(length_rows)

    overall_df = overall_df.sort_values(
        by=["dataset_name", "variant_name", "evaluation_scope", "max_accuracy", "roc_auc"],
        ascending=[True, True, True, False, False],
        na_position="last",
    ).reset_index(drop=True)

    length_df = length_df.sort_values(
        by=["dataset_name", "variant_name", "question_length_group", "response_length_group"],
        ascending=True,
        na_position="last",
    ).reset_index(drop=True)

    overall_df.to_csv(overall_csv, index=False)
    length_df.to_csv(length_csv, index=False)

    sgi_cfg_payload = read_json(sgi_cfg, default={})
    write_json(
        config_json,
        {
            "dataset_name": dataset_name,
            "variant_name": variant_name,
            "model_name": MODEL_NAME,
            "cache_version": CACHE_VERSION,
            "rows_used": int(len(eval_df)),
            "length_mode": LENGTH_MODE,
            "length_split_mode": LENGTH_SPLIT_MODE,
            "default_threshold": float(DEFAULT_SGI_THRESHOLD),
            "source_sgi_csv": str(sgi_csv),
            "source_sgi_signature": sgi_cfg_payload,
        },
    )

    print(f"✅ Evaluation saved: {dataset_name} | rows={len(eval_df)}")
    print("   overall_csv:", overall_csv)
    print("   length_csv :", length_csv)

    return {
        "dataset_name": dataset_name,
        "variant_name": variant_name,
        "status": "computed",
        "rows_used": int(len(eval_df)),
        "overall_csv": str(overall_csv),
        "length_csv": str(length_csv),
    }


def run_evaluation_pipeline(dataset_name: str, variant_name: str = "complete_triplets", overwrite: bool = False) -> dict:
    return evaluate_sgi_csv(
        dataset_name=dataset_name,
        variant_name=variant_name,
        overwrite=overwrite,
    )


def get_evaluation_status() -> pd.DataFrame:
    rows = []
    base = EVAL_DIR
    if not base.exists():
        return pd.DataFrame(columns=[
            "dataset_name", "variant_name", "overall_exists", "length_exists", "rows_used",
            "overall_csv", "length_csv"
        ])

    for dataset_dir in sorted(base.iterdir()):
        if not dataset_dir.is_dir():
            continue
        for variant_dir in sorted(dataset_dir.iterdir()):
            if not variant_dir.is_dir():
                continue
            overall_csv = variant_dir / "evaluation_overall.csv"
            length_csv = variant_dir / "evaluation_length_splits.csv"
            config_json = variant_dir / "config.json"
            cfg = read_json(config_json, default={})
            rows.append({
                "dataset_name": dataset_dir.name,
                "variant_name": variant_dir.name,
                "overall_exists": overall_csv.exists(),
                "length_exists": length_csv.exists(),
                "rows_used": cfg.get("rows_used", np.nan),
                "overall_csv": str(overall_csv) if overall_csv.exists() else "",
                "length_csv": str(length_csv) if length_csv.exists() else "",
            })

    return pd.DataFrame(rows)


display(get_evaluation_status())


## Uruchomienie ewaluacji dla wybranych datasetów

In [ ]:
halueval_eval_result = run_evaluation_pipeline("HaluEval")
halueval_eval_result

In [ ]:
medhal_eval_result = run_evaluation_pipeline("MedHal")
medhal_eval_result

In [ ]:
medhallu_eval_result = run_evaluation_pipeline("MedHallu")
medhallu_eval_result

## Status plików ewaluacyjnych po uruchomieniu

In [ ]:
display(get_evaluation_status())

# Komórka 7: Agregacja wyników i raportowanie

Ta sekcja łączy cząstkowe pliki ewaluacyjne w jeden raport końcowy i porządkuje go zgodnie z hierarchią:

**Parametry → Dataset → Modele**

Raport końcowy zawiera między innymi:
- `optimal_threshold`
- `max_accuracy`
- `roc_auc`


In [ ]:

from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

FINAL_REPORT_DIR = PROJECT_ROOT / "reports"
FINAL_REPORT_DIR.mkdir(parents=True, exist_ok=True)


def collect_evaluation_files(base_dir: Path = PROJECT_ROOT / "evaluation") -> list[Path]:
    if not base_dir.exists():
        return []
    return sorted(base_dir.rglob("evaluation_overall.csv"))


def load_and_annotate_evaluation_csv(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path)

    # Ścieżka: evaluation / cache_version / model_slug / dataset_name / variant_name / evaluation_overall.csv
    parts = csv_path.parts
    try:
        idx = parts.index("evaluation")
        cache_version = parts[idx + 1]
        model_slug = parts[idx + 2]
        dataset_name_from_path = parts[idx + 3]
        variant_name_from_path = parts[idx + 4]
    except Exception:
        cache_version = ""
        model_slug = ""
        dataset_name_from_path = ""
        variant_name_from_path = ""

    if "cache_version" not in df.columns:
        df["cache_version"] = cache_version
    else:
        df["cache_version"] = df["cache_version"].fillna(cache_version)

    if "model_slug" not in df.columns:
        df["model_slug"] = model_slug
    else:
        df["model_slug"] = df["model_slug"].fillna(model_slug)

    if "dataset_name" not in df.columns:
        df["dataset_name"] = dataset_name_from_path
    else:
        df["dataset_name"] = df["dataset_name"].fillna(dataset_name_from_path)

    if "variant_name" not in df.columns:
        df["variant_name"] = variant_name_from_path
    else:
        df["variant_name"] = df["variant_name"].fillna(variant_name_from_path)

    df["source_file"] = str(csv_path)
    return df


def build_final_report(base_dir: Path = PROJECT_ROOT / "evaluation") -> dict:
    files = collect_evaluation_files(base_dir=base_dir)
    if not files:
        raise FileNotFoundError(f"Nie znaleziono żadnych plików evaluation_overall.csv w {base_dir}")

    parts = [load_and_annotate_evaluation_csv(path) for path in files]
    report_df = pd.concat(parts, ignore_index=True)

    # Zostawiamy raport ogólny oraz ewentualne przekroje po parametrach.
    report_df = report_df[
        report_df["evaluation_scope"].isin(["overall", "parameter_slice"])
    ].copy()

    report_df["parameter_signature"] = (
        "variant=" + report_df["variant_name"].fillna("unknown").astype(str)
        + " | question_mode=" + report_df["question_mode"].fillna("mixed").astype(str)
        + " | sgi_setting=" + report_df["sgi_setting"].fillna("mixed").astype(str)
    )

    preferred_columns = [
        "parameter_signature",
        "variant_name",
        "question_mode",
        "sgi_setting",
        "dataset_name",
        "model_name",
        "model_slug",
        "cache_version",
        "evaluation_scope",
        "default_threshold",
        "optimal_threshold",
        "accuracy_at_default_threshold",
        "max_accuracy",
        "roc_auc",
        "n_samples",
        "n_pos",
        "n_neg",
        "positive_rate",
        "source_file",
    ]
    existing = [c for c in preferred_columns if c in report_df.columns]
    remaining = [c for c in report_df.columns if c not in existing]
    report_df = report_df[existing + remaining].copy()

    report_df = report_df.sort_values(
        by=["parameter_signature", "dataset_name", "model_name", "max_accuracy", "roc_auc"],
        ascending=[True, True, True, False, False],
        na_position="last",
    ).reset_index(drop=True)

    out_csv = FINAL_REPORT_DIR / "final_report.csv"
    report_df.to_csv(out_csv, index=False)

    print("✅ Final report saved:", out_csv)
    return {
        "rows": int(len(report_df)),
        "n_files": int(len(files)),
        "final_report_csv": str(out_csv),
    }


final_report_info = build_final_report()
display(pd.read_csv(final_report_info["final_report_csv"]).head(20))


# Komórka 8: Podsumowanie analityczne (Złoty Standard)

Ta sekcja analizuje raport końcowy i wybiera najlepsze konfiguracje parametryczne na poziomie datasetu.

Zasada:
- agregujemy wyniki po **dataset_name + parametrach**,
- uśredniamy wyniki po dostępnych modelach,
- wybieramy konfigurację o najwyższym `avg_max_accuracy`,
- przy remisie patrzymy na `avg_roc_auc`.


In [ ]:

from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

GOLD_REPORT_DIR = PROJECT_ROOT / "gold_summary"
GOLD_REPORT_DIR.mkdir(parents=True, exist_ok=True)


def build_gold_summary(final_report_csv: Path | str = FINAL_REPORT_DIR.parent / "reports" / "final_report.csv") -> dict:
    final_report_csv = Path(final_report_csv)
    if not final_report_csv.exists():
        raise FileNotFoundError(f"Brak final_report.csv: {final_report_csv}")

    report_df = pd.read_csv(final_report_csv)
    if len(report_df) == 0:
        raise ValueError("final_report.csv jest pusty.")

    metric_cols = ["optimal_threshold", "max_accuracy", "roc_auc"]
    for col in metric_cols:
        if col in report_df.columns:
            report_df[col] = pd.to_numeric(report_df[col], errors="coerce")

    group_cols = ["dataset_name", "variant_name", "question_mode", "sgi_setting"]
    grouped = (
        report_df
        .groupby(group_cols, dropna=False)
        .agg(
            n_rows=("dataset_name", "size"),
            n_models=("model_name", "nunique"),
            models=("model_name", lambda x: " | ".join(sorted({str(v) for v in x.dropna()}))),
            avg_optimal_threshold=("optimal_threshold", "mean"),
            std_optimal_threshold=("optimal_threshold", "std"),
            avg_max_accuracy=("max_accuracy", "mean"),
            std_max_accuracy=("max_accuracy", "std"),
            avg_roc_auc=("roc_auc", "mean"),
            std_roc_auc=("roc_auc", "std"),
        )
        .reset_index()
    )

    grouped["parameter_signature"] = (
        "variant=" + grouped["variant_name"].fillna("unknown").astype(str)
        + " | question_mode=" + grouped["question_mode"].fillna("mixed").astype(str)
        + " | sgi_setting=" + grouped["sgi_setting"].fillna("mixed").astype(str)
    )

    grouped = grouped.sort_values(
        by=["dataset_name", "avg_max_accuracy", "avg_roc_auc"],
        ascending=[True, False, False],
        na_position="last",
    ).reset_index(drop=True)

    aggregated_csv = GOLD_REPORT_DIR / "gold_aggregated_candidates.csv"
    grouped.to_csv(aggregated_csv, index=False)

    idx_best = (
        grouped.groupby("dataset_name", dropna=False)["avg_max_accuracy"]
        .transform("max")
        == grouped["avg_max_accuracy"]
    )
    best_candidates = grouped[idx_best].copy()
    best_candidates = best_candidates.sort_values(
        by=["dataset_name", "avg_roc_auc", "avg_optimal_threshold"],
        ascending=[True, False, True],
        na_position="last",
    ).groupby("dataset_name", dropna=False, as_index=False).head(1).reset_index(drop=True)

    best_candidates["gold_conclusion"] = (
        "Rekomendowana konfiguracja dla datasetu "
        + best_candidates["dataset_name"].astype(str)
        + ": "
        + best_candidates["parameter_signature"].astype(str)
        + " | Avg. Max Accuracy="
        + (100 * best_candidates["avg_max_accuracy"]).round(2).astype(str) + "%"
        + " | Avg. ROC-AUC="
        + best_candidates["avg_roc_auc"].round(4).astype(str)
        + " | Avg. Optymalny próg="
        + best_candidates["avg_optimal_threshold"].round(4).astype(str)
    )

    gold_csv = GOLD_REPORT_DIR / "gold_summary.csv"
    best_candidates.to_csv(gold_csv, index=False)

    print("✅ Aggregated candidates saved:", aggregated_csv)
    print("✅ Gold summary saved        :", gold_csv)

    return {
        "aggregated_candidates_csv": str(aggregated_csv),
        "gold_summary_csv": str(gold_csv),
        "n_datasets": int(best_candidates["dataset_name"].nunique()),
    }


gold_summary_info = build_gold_summary()
display(pd.read_csv(gold_summary_info["gold_summary_csv"]))
